# Adaptive Confidence-Based Cascaded Fusion for Multi-Modal Webpage Phishing Detection

This notebook reproduces all experiments of the paper. The URL, HTML text, and screenshot of a webpage are evaluated **in order of computational cost**, and the available evidence is **weighted per sample by its confidence**, so a page exits the cascade as soon as the decision is sufficiently certain.

**Pipeline**

1. Environment setup and configuration
2. Dataset loading and stratified 70/15/15 split
3. Preprocessing and feature extraction per modality
4. Modality-specific classifiers (Linear SVM for URL and HTML, MobileNetV3-Small + RBF SVM for the screenshot) and modality ablation
5. Static fusion baselines (early fusion, majority voting, average and weighted average, stacking, hybrid)
6. Proposed method (static cascade, dynamic weighting, and cascade + dynamic weighting)
7. Final evaluation on the test set
8. Efficiency evaluation on CPU (batch size 1)
9. Visualization of the proposed method

**How to run.** Open the notebook in Google Colab (a high-RAM runtime with a T4 GPU was used in the paper) and run all cells in order. The GPU is only used for training, and every latency is measured on the CPU. All results are saved in the `outputs/` folder.

**Dataset.** Lee et al., *Multimodal Large Language Models for Phishing Webpage Detection and Identification*, 2024. https://github.com/JehLeeKR/Multimodal_LLM_Phishing_Detection

**Reproducibility.** All random processes use `SEED = 42`. Every hyperparameter, fusion weight, and threshold is selected on the validation set only, and the test set is evaluated once at the end.

# 1. Environment Setup

In [ ]:
# Install extra packages (most of them are preinstalled on Google Colab)
!pip install -q tqdm pillow psutil

import os

# Clone the public dataset of Lee et al. (2024)
if not os.path.exists('Multimodal_LLM_Phishing_Detection'):
    !git clone --depth 1 https://github.com/JehLeeKR/Multimodal_LLM_Phishing_Detection.git

# Standard library
import re, gc, io, json, math, time, platform, warnings, zipfile, ipaddress
from pathlib import Path
from collections import Counter
from contextlib import contextmanager
from itertools import combinations, product
from urllib.parse import urlparse
from concurrent.futures import ThreadPoolExecutor

# Scientific stack
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from PIL import Image
from scipy import sparse
import psutil, joblib

# Deep learning (frozen image feature extractor)
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.applications import MobileNetV3Small

# Machine learning
import sklearn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.svm import LinearSVC, SVC
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, roc_curve,
                             precision_recall_curve, average_precision_score, auc as sk_auc)

warnings.filterwarnings('ignore')

# Reproducibility
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
keras.utils.set_random_seed(SEED)   # seeds Python, NumPy, and TensorFlow at once

print('Python      :', platform.python_version())
print('scikit-learn:', sklearn.__version__)
print('TensorFlow  :', tf.__version__)
print('GPU         :', tf.config.list_physical_devices('GPU'))

# 2. Configuration and Helper Functions

In [ ]:
CFG = {
    'DATA_DIR': Path('Multimodal_LLM_Phishing_Detection'),
    'OUT_DIR':  Path('outputs'),

    # Stratified 70/15/15 split
    'VAL_SIZE': 0.15, 'TEST_SIZE': 0.15,

    # URL: character n-gram TF-IDF
    'URL_TFIDF':  dict(analyzer='char', ngram_range=(1, 3), max_features=50_000, sublinear_tf=True),
    # HTML: word unigram and bigram TF-IDF over the extracted page text
    'HTML_TFIDF': dict(analyzer='word', ngram_range=(1, 2), max_features=50_000,
                       sublinear_tf=True, token_pattern=r'[^\s]+'),

    # Screenshot: input resolution of MobileNetV3-Small
    'IMG_SIZE': (224, 224),

    # Early fusion: TF-IDF blocks are reduced with SVD before concatenation
    'SVD_DIM': 256,

    # Probability calibration of every SVM (Platt scaling)
    'CALIB_METHOD': 'sigmoid',

    # Cascade threshold grid. A page exits if p >= tau_phish or p <= 1 - tau_legit
    'TAU_GRID': np.round(np.arange(0.70, 1.00, 0.025), 3),
}
CFG['OUT_DIR'].mkdir(exist_ok=True)
FIG_DIR = CFG['OUT_DIR'] / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

RESULTS = []   # validation results of every configuration


def evaluate(y_true, y_prob, thr=0.5):
    """Detection metrics, with phishing as the positive class."""
    y_pred = (y_prob >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return dict(accuracy=accuracy_score(y_true, y_pred),
                precision=precision_score(y_true, y_pred, zero_division=0),
                recall=recall_score(y_true, y_pred, zero_division=0),
                f1=f1_score(y_true, y_pred, zero_division=0),
                fpr=fp / max(fp + tn, 1),
                auc=roc_auc_score(y_true, y_prob))


@contextmanager
def measure(store: dict):
    """Measure the wall time and RAM usage of a code block."""
    proc = psutil.Process()
    gc.collect()
    mem0, t0 = proc.memory_info().rss, time.perf_counter()
    yield
    store['time_s'] = time.perf_counter() - t0
    store['ram_mb'] = (proc.memory_info().rss - mem0) / 1e6


def log_result(name, group, metrics, eff=None, n_samples=None, extra=None):
    """Append one result row to RESULTS."""
    row = {'config': name, 'group': group, **metrics}
    if eff:
        row.update(eff)
        if n_samples:
            row['latency_ms_per_sample'] = 1000 * eff['time_s'] / n_samples
    if extra:
        row.update(extra)
    RESULTS.append(row)


print('Configuration ready.')

# 3. Dataset Loading and Split

In [ ]:
BENIGN_DIR   = CFG['DATA_DIR'] / 'data' / 'MMLLM_Benign'
PHISHING_DIR = CFG['DATA_DIR'] / 'data' / 'MMLLM_Phishing'
EXTRACT_DIR  = Path('extracted')          # ZIP files are extracted here once

# 'html_brand_info' is NOT raw HTML. It holds text from nine page elements.
HTML_FIELDS = ['title', 'meta_description', 'favicon', 'logo_alt_text', 'footer_text',
               'headers_text', 'nav_bar_content', 'paragraphs_text', 'span_text']


def parse_brand_info(text: str) -> dict:
    """Split html_brand_info into the nine fields. 'Not Found' is treated as empty."""
    out, cur = {f: '' for f in HTML_FIELDS}, None
    for line in text.split('\n'):
        key = line.split(':', 1)[0].strip()
        if key in out and ':' in line:
            cur = key
            out[cur] = line.split(':', 1)[1].strip()
        elif cur:                                   # continuation of a multi-line value
            out[cur] += ' ' + line.strip()
    return {k: ('' if v.strip() == 'Not Found' else v.strip()) for k, v in out.items()}


def extract_all(src_dir: Path, name: str) -> list:
    """Extract every ZIP file (idempotent). Returns a list of (brand, sample_dir)."""
    samples = []
    for z in tqdm(sorted(src_dir.rglob('*.zip')), desc=f'Extract {name}'):
        out = EXTRACT_DIR / name / z.relative_to(src_dir).with_suffix('')
        if not out.exists():
            try:
                with zipfile.ZipFile(z) as zf:
                    zf.extractall(out)
            except zipfile.BadZipFile:
                print('  Skipped (bad zip):', z)
                continue
        samples += [(z.stem, p.parent) for p in out.rglob('add_info.json')]
    return samples


def build_dataframe() -> pd.DataFrame:
    rows = []
    for label, src, name in [(0, BENIGN_DIR, 'benign'), (1, PHISHING_DIR, 'phishing')]:
        for brand, sdir in extract_all(src, name):
            try:
                info = json.loads((sdir / 'add_info.json').read_text(encoding='utf-8', errors='ignore'))
            except json.JSONDecodeError:
                continue
            img = sdir / 'screenshot_aft.png'
            rows.append(dict(
                sample_id=sdir.name, brand=brand, label=label,
                url=str(info.get('Url', '')).strip(),               # original page URL
                html_text=str(info.get('html_brand_info', '')).strip(),
                screenshot_path=str(img) if img.exists() else None,
            ))
    df = pd.DataFrame(rows)
    fields = pd.DataFrame(df['html_text'].map(parse_brand_info).tolist())
    df = pd.concat([df, fields.add_prefix('html_')], axis=1)

    # A screenshot only counts as available if it can be opened
    def img_ok(p):
        if p is None:
            return False
        try:
            with Image.open(p) as im:
                im.verify()
            return True
        except Exception:
            return False

    df['has_url']  = df['url'].str.len() > 0
    df['has_html'] = df['html_text'].str.len() > 0
    df['has_img']  = [img_ok(p) for p in tqdm(df['screenshot_path'], desc='Verify images')]
    df['complete'] = df['has_url'] & df['has_html'] & df['has_img']
    return df


# --- Load and clean ---
df_all = build_dataframe()

conflict = df_all.groupby('url')['label'].transform('nunique') > 1   # same URL, different labels
print(f'Removed {conflict.sum()} rows with conflicting labels')
df_all = df_all[~conflict].drop_duplicates('url').reset_index(drop=True)

print('\nModality availability per class:')
print(df_all.groupby('label')[['has_url', 'has_html', 'has_img', 'complete']].sum()
      .rename(index={0: 'benign', 1: 'phishing'}))

# Main experiments use only samples with all three modalities
df = df_all[df_all['complete']].reset_index(drop=True)
df_missing = df_all[~df_all['complete']].reset_index(drop=True)   # kept for reference only

# --- Stratified 70/15/15 split ---
idx_train, idx_tmp = train_test_split(df.index, test_size=CFG['VAL_SIZE'] + CFG['TEST_SIZE'],
                                      stratify=df['label'], random_state=SEED)
rel_test = CFG['TEST_SIZE'] / (CFG['VAL_SIZE'] + CFG['TEST_SIZE'])
idx_val, idx_test = train_test_split(idx_tmp, test_size=rel_test,
                                     stratify=df.loc[idx_tmp, 'label'], random_state=SEED)
df['split'] = 'train'
df.loc[idx_val, 'split'] = 'val'
df.loc[idx_test, 'split'] = 'test'

print(f'\nMain dataset: {len(df)} samples | phishing {df.label.mean():.1%}')
print(pd.crosstab(df['split'], df['label'].map({0: 'benign', 1: 'phishing'}), margins=True))
print(f'Incomplete samples (not used): {len(df_missing)}')

df.to_pickle(CFG['OUT_DIR'] / 'dataset_complete.pkl')
df_missing.to_pickle(CFG['OUT_DIR'] / 'dataset_incomplete.pkl')
df[['url', 'html_title', 'label', 'split']].head()

# 4. Preprocessing and Feature Extraction

All transformations (scalers, TF-IDF vocabularies, SVD) are fitted on the **training set only** to prevent leakage.

## 4.1 URL

In [ ]:
df_train = df[df['split'] == 'train'].reset_index(drop=True)
df_val   = df[df['split'] == 'val'].reset_index(drop=True)
df_test  = df[df['split'] == 'test'].reset_index(drop=True)
y_train, y_val, y_test = df_train['label'].values, df_val['label'].values, df_test['label'].values

# --- (a) Cleaning and normalization ---
def clean_url(url: str) -> str:
    """Strip whitespace, add a scheme if missing, and lowercase the hostname."""
    url = url.strip()
    if not re.match(r'^[a-zA-Z][a-zA-Z0-9+.-]*://', url):
        url = 'http://' + url
    p = urlparse(url)
    return p._replace(netloc=p.netloc.lower()).geturl()

def url_for_tfidf(url: str) -> str:
    """TF-IDF input without the scheme and 'www.', so the model learns from the URL structure
    rather than from HTTPS (HTTPS is kept as a separate handcrafted feature)."""
    return re.sub(r'^[a-z]+://(www\.)?', '', url.lower())

# --- (b) Handcrafted lexical features ---
SUSPICIOUS_KW = ['login', 'verify', 'secure', 'account', 'update', 'confirm', 'bank',
                 'paypal', 'password', 'signin', 'authenticate', 'wallet', 'support']
FREE_HOSTING  = ['replit.app', 'brizy.site', 'weebly.com', 'wixsite.com', '000webhostapp.com',
                 'firebaseapp.com', 'web.app', 'github.io', 'netlify.app', 'vercel.app',
                 'glitch.me', 'pages.dev', 'blogspot.com', 'sites.google.com', 'workers.dev']
SHORTENERS    = ['bit.ly', 'tinyurl.com', 'goo.gl', 't.co', 'ow.ly', 'is.gd', 'cutt.ly', 's.id']

def entropy(s: str) -> float:
    """Shannon entropy of a string (bits)."""
    if not s:
        return 0.0
    n = len(s)
    return -sum(c / n * math.log2(c / n) for c in Counter(s).values())

def is_ip(host: str) -> int:
    try:
        ipaddress.ip_address(host)
        return 1
    except ValueError:
        return 0

def has_port(p) -> int:
    """1 if the URL has an explicit port. A malformed netloc (e.g. 'https://https://') also counts as 1."""
    try:
        return int(p.port is not None)
    except ValueError:
        return 1

def extract_url_features(url: str) -> dict:
    p    = urlparse(url)
    host = (p.hostname or '')
    path = p.path or ''
    low  = url.lower()
    n    = max(len(url), 1)
    return {
        # Length
        'url_length':      len(url),
        'host_length':     len(host),
        'path_length':     len(path),
        'query_length':    len(p.query),
        # Structure
        'num_subdomains':  max(host.count('.') - 1, 0),
        'path_depth':      path.count('/'),
        'num_dots':        url.count('.'),
        'num_hyphens':     url.count('-'),
        'num_params':      url.count('='),
        'has_at_sign':     int('@' in url),
        'has_port':        has_port(p),
        'has_ip_host':     is_ip(host),
        'has_https':       int(p.scheme == 'https'),
        # Character composition
        'digit_ratio':     sum(c.isdigit() for c in url) / n,
        'special_ratio':   sum(not c.isalnum() for c in url) / n,
        'entropy':         entropy(low),
        'host_entropy':    entropy(host),
        # Lexical and reputation cues
        'suspicious_count': sum(kw in low for kw in SUSPICIOUS_KW),
        'is_free_hosting': int(any(host == d or host.endswith('.' + d) for d in FREE_HOSTING)),
        'is_shortener':    int(host in SHORTENERS),
    }

# --- (c) Extraction (time is measured for the cost of cascade stage 1) ---
for d in (df_train, df_val, df_test):
    d['url_clean'] = d['url'].map(clean_url)

eff_url_hc = {}
with measure(eff_url_hc):
    H_url_train = pd.DataFrame([extract_url_features(u) for u in df_train['url_clean']])
H_url_val  = pd.DataFrame([extract_url_features(u) for u in df_val['url_clean']])
H_url_test = pd.DataFrame([extract_url_features(u) for u in df_test['url_clean']])

url_scaler = StandardScaler()
H_url_train_s = url_scaler.fit_transform(H_url_train)
H_url_val_s   = url_scaler.transform(H_url_val)
H_url_test_s  = url_scaler.transform(H_url_test)

# --- (d) Character n-gram TF-IDF (kept sparse) ---
url_tfidf = TfidfVectorizer(**CFG['URL_TFIDF'], min_df=2, dtype=np.float32)
T_url_train = url_tfidf.fit_transform(df_train['url_clean'].map(url_for_tfidf))
T_url_val   = url_tfidf.transform(df_val['url_clean'].map(url_for_tfidf))
T_url_test  = url_tfidf.transform(df_test['url_clean'].map(url_for_tfidf))

# --- (e) Concatenate handcrafted + TF-IDF features ---
X_url_train = sparse.hstack([sparse.csr_matrix(H_url_train_s), T_url_train]).tocsr()
X_url_val   = sparse.hstack([sparse.csr_matrix(H_url_val_s),   T_url_val]).tocsr()
X_url_test  = sparse.hstack([sparse.csr_matrix(H_url_test_s),  T_url_test]).tocsr()

URL_FEATURE_NAMES = list(H_url_train.columns) + list(url_tfidf.get_feature_names_out())
joblib.dump({'scaler': url_scaler, 'tfidf': url_tfidf}, CFG['OUT_DIR'] / 'url_preproc.joblib')

print(f'URL feature matrix (train): {X_url_train.shape} | sparsity {1 - X_url_train.nnz / np.prod(X_url_train.shape):.1%}')
print(f'  Handcrafted: {H_url_train.shape[1]} | TF-IDF: {T_url_train.shape[1]}')
print(f'  Handcrafted extraction time: {1000 * eff_url_hc["time_s"] / len(df_train):.3f} ms/URL')

# Sanity check: mean of each handcrafted feature per class
print('\nMean handcrafted features per class (train):')
print(H_url_train.groupby(y_train).mean().T.rename(columns={0: 'benign', 1: 'phishing'}).round(3))

## 4.2 HTML

In [ ]:
TEXT_FIELDS = [f for f in HTML_FIELDS if f != 'favicon']        # favicon is a path/URL, not text
ITEM_FIELDS = ['headers_text', 'nav_bar_content', 'paragraphs_text', 'span_text']  # items separated by ' | '
FIELD_CAP   = 1000   # character limit per field, limits the dominance of very long pages

# Multilingual login/credential keywords (the dataset contains EN, ES, PT, FR, DE, JA, ZH, ID, ...)
LOGIN_KW = ['login', 'log in', 'sign in', 'signin', 'password', 'verify', 'verification',
            'account', 'iniciar sesión', 'contraseña', 'usuario', 'ingresar', 'senha', 'entrar',
            'connexion', 'mot de passe', 'anmelden', 'passwort', 'masuk', 'kata sandi',
            'ログイン', 'パスワード', '登录', '密码', '로그인', '비밀번호']

# --- (a) Text cleaning ---
def clean_field(s: str) -> str:
    s = re.sub(r'\s+', ' ', s).strip()
    return s[:FIELD_CAP]

def html_for_tfidf(row) -> str:
    """Merge the text fields (without favicon) into one lowercase document."""
    return ' '.join(clean_field(row[f'html_{f}']) for f in TEXT_FIELDS).lower()

# --- (b) Structural features of the page elements ---
def favicon_type(fav: str) -> dict:
    fav = fav.strip().lower()
    return {
        'fav_empty':    int(fav == ''),
        'fav_data_uri': int(fav.startswith('data:')),                 # base64-embedded icon
        'fav_absolute': int(fav.startswith(('http://', 'https://', '//'))),
        'fav_relative': int(fav != '' and not fav.startswith(('data:', 'http://', 'https://', '//'))),
    }

def extract_html_features(row) -> dict:
    feats = {}
    texts = {f: row[f'html_{f}'] for f in HTML_FIELDS}

    # Completeness and (log) length of each field
    for f in HTML_FIELDS:
        feats[f'{f}_empty']   = int(texts[f] == '')
        feats[f'{f}_len_log'] = math.log1p(len(texts[f]))
    feats['n_empty_fields'] = sum(feats[f'{f}_empty'] for f in HTML_FIELDS)
    feats['total_len_log']  = math.log1p(sum(len(texts[f]) for f in TEXT_FIELDS))

    # Number of items (proxy for the number of <h*>, <nav>, <p>, and <span> elements)
    for f in ITEM_FIELDS:
        feats[f'{f}_n_items'] = 0 if texts[f] == '' else texts[f].count(' | ') + 1

    # Favicon
    feats.update(favicon_type(texts['favicon']))

    # Content
    body = ' '.join(texts[f] for f in TEXT_FIELDS).lower()
    feats['login_kw_count'] = sum(kw in body for kw in LOGIN_KW)
    feats['title_login_kw'] = int(any(kw in texts['title'].lower() for kw in LOGIN_KW))
    feats['has_copyright']  = int(bool(re.search(r'©|copyright|&copy;', texts['footer_text'].lower())))
    feats['non_ascii_ratio'] = sum(ord(c) > 127 for c in body) / max(len(body), 1)
    return feats

# --- (c) Extraction (time is measured for the cost of cascade stage 2) ---
eff_html_hc = {}
with measure(eff_html_hc):
    H_html_train = pd.DataFrame([extract_html_features(r) for _, r in df_train.iterrows()])
H_html_val  = pd.DataFrame([extract_html_features(r) for _, r in df_val.iterrows()])
H_html_test = pd.DataFrame([extract_html_features(r) for _, r in df_test.iterrows()])

html_scaler = StandardScaler()
H_html_train_s = html_scaler.fit_transform(H_html_train)
H_html_val_s   = html_scaler.transform(H_html_val)
H_html_test_s  = html_scaler.transform(H_html_test)

# --- (d) TF-IDF over the merged page text (see CFG['HTML_TFIDF'], kept sparse) ---
docs_train = df_train.apply(html_for_tfidf, axis=1)
docs_val   = df_val.apply(html_for_tfidf, axis=1)
docs_test  = df_test.apply(html_for_tfidf, axis=1)

html_tfidf = TfidfVectorizer(**CFG['HTML_TFIDF'], dtype=np.float32)
T_html_train = html_tfidf.fit_transform(docs_train)
T_html_val   = html_tfidf.transform(docs_val)
T_html_test  = html_tfidf.transform(docs_test)

# --- (e) Concatenate structural + TF-IDF features ---
X_html_train = sparse.hstack([sparse.csr_matrix(H_html_train_s), T_html_train]).tocsr()
X_html_val   = sparse.hstack([sparse.csr_matrix(H_html_val_s),   T_html_val]).tocsr()
X_html_test  = sparse.hstack([sparse.csr_matrix(H_html_test_s),  T_html_test]).tocsr()

HTML_FEATURE_NAMES = list(H_html_train.columns) + list(html_tfidf.get_feature_names_out())
joblib.dump({'scaler': html_scaler, 'tfidf': html_tfidf}, CFG['OUT_DIR'] / 'html_preproc.joblib')

print(f'HTML feature matrix (train): {X_html_train.shape} | sparsity {1 - X_html_train.nnz / np.prod(X_html_train.shape):.1%}')
print(f'  Structural: {H_html_train.shape[1]} | TF-IDF: {T_html_train.shape[1]}')
print(f'  Structural extraction time: {1000 * eff_html_hc["time_s"] / len(df_train):.3f} ms/sample')

print('\nMean of selected structural features per class (train):')
cols = ['n_empty_fields', 'total_len_log', 'paragraphs_text_n_items', 'fav_data_uri',
        'fav_relative', 'login_kw_count', 'title_login_kw', 'has_copyright', 'non_ascii_ratio']
print(H_html_train[cols].groupby(y_train).mean().T.rename(columns={0: 'benign', 1: 'phishing'}).round(3))

## 4.3 Screenshot

In [ ]:
Image.MAX_IMAGE_PIXELS = None            # some full-page screenshots exceed 89 million pixels
VIEWPORT_RATIO = 720 / 1280              # height/width of a desktop viewport
IMG_SIZE_MB = CFG['IMG_SIZE']            # 224 x 224, the input size of MobileNetV3-Small

def preprocess_image(path: str, size=CFG['IMG_SIZE']) -> np.ndarray:
    """RGBA -> RGB (white background), crop the first viewport, resize. Output uint8 (H, W, 3)."""
    with Image.open(path) as im:
        w = im.size[0]
        im = im.crop((0, 0, w, int(round(w * VIEWPORT_RATIO))))       # first viewport
        if im.mode in ('RGBA', 'LA', 'P'):
            im = im.convert('RGBA')
            bg = Image.new('RGB', im.size, (255, 255, 255))
            bg.paste(im, mask=im.split()[-1])                          # transparent -> white
            im = bg
        else:
            im = im.convert('RGB')
        im = im.resize((size[1], size[0]), Image.BILINEAR)            # PIL expects (W, H)
        return np.asarray(im, dtype=np.uint8)

def load_split_cached(d: pd.DataFrame, name: str, size=CFG['IMG_SIZE']) -> np.ndarray:
    """Preprocess all screenshots of a split once and cache them as .npy."""
    f = CFG['OUT_DIR'] / f'img_{name}_{size[0]}x{size[1]}.npy'
    if f.exists():
        return np.load(f)
    paths = d['screenshot_path'].tolist()
    with ThreadPoolExecutor(8) as ex:
        arr = np.stack(list(tqdm(ex.map(lambda p: preprocess_image(p, size=size), paths),
                                 total=len(paths), desc=f'Images {name}')))
    np.save(f, arr)
    return arr

X224_train = load_split_cached(df_train, 'train')
X224_val   = load_split_cached(df_val,   'val')
X224_test  = load_split_cached(df_test,  'test')

# Preprocessing cost per image (measured without the cache)
eff_img_pre = {}
sample_paths = df_test['screenshot_path'].sample(100, random_state=SEED).tolist()
with measure(eff_img_pre):
    _ = [preprocess_image(p) for p in sample_paths]
img_pre_ms = 1000 * eff_img_pre['time_s'] / len(sample_paths)

print(f'Image tensors: train={X224_train.shape}, val={X224_val.shape}, test={X224_test.shape}')
print(f'Memory: {(X224_train.nbytes + X224_val.nbytes + X224_test.nbytes) / 1e6:.0f} MB (uint8)')
print(f'Preprocessing time: {img_pre_ms:.1f} ms/image')

# Visual check: 4 benign (top) vs 4 phishing (bottom)
rng = np.random.default_rng(SEED)
fig, axes = plt.subplots(2, 4, figsize=(16, 6))
for r, lab in enumerate([0, 1]):
    for ax, i in zip(axes[r], rng.choice(np.where(y_train == lab)[0], 4, replace=False)):
        ax.imshow(X224_train[i])
        ax.set_title(f'{"Phishing" if lab else "Benign"} | {df_train.loc[i, "brand"]}', fontsize=9)
        ax.axis('off')
plt.suptitle('Sample screenshots (first viewport, after preprocessing)')
plt.tight_layout()
plt.savefig(FIG_DIR / 'sample_screenshots.png', dpi=150)
plt.show()

# 5. Modality-Specific Classifiers

Each modality has its own classifier. Hyperparameters are selected by the highest F1-score on the validation set, and the outputs are calibrated with Platt scaling (five-fold CV on the training set).

## 5.1 URL Only (Linear SVM)

In [ ]:
P_val, P_test = {}, {}          # phishing probabilities per modality, reused by every fusion method

# --- (a) Select C on the validation set ---
C_GRID = [0.01, 0.03, 0.1, 0.3, 1, 3, 10]
f1_per_c = {}
for C in C_GRID:
    m = LinearSVC(C=C, max_iter=5000, random_state=SEED).fit(X_url_train, y_train)
    f1_per_c[C] = f1_score(y_val, m.predict(X_url_val))
best_C = max(f1_per_c, key=f1_per_c.get)
print('Validation F1 per C:', {c: round(v, 4) for c, v in f1_per_c.items()})
print(f'Selected C: {best_C}')

# --- (b) Final model + probability calibration (5-fold CV on TRAIN) ---
eff_url_train = {}
with measure(eff_url_train):
    svm_url = CalibratedClassifierCV(
        LinearSVC(C=best_C, max_iter=5000, random_state=SEED),
        method=CFG['CALIB_METHOD'], cv=5).fit(X_url_train, y_train)
print(f'Training time: {eff_url_train["time_s"]:.2f} s')

# --- (c) Validation results (test probabilities are stored for Section 8) ---
P_val['url']  = svm_url.predict_proba(X_url_val)[:, 1]
P_test['url'] = svm_url.predict_proba(X_url_test)[:, 1]

m_val = evaluate(y_val, P_val['url'])
print('\nURL only (SVM), validation:', {k: round(v, 4) for k, v in m_val.items()})
print('Confusion matrix (rows = true, columns = predicted) [benign, phishing]:')
print(confusion_matrix(y_val, (P_val['url'] >= 0.5).astype(int)))

# --- (d) End-to-end latency per URL (batch size 1) ---
#     cleaning -> handcrafted features -> TF-IDF -> probability
def predict_url_single(url: str) -> float:
    u = clean_url(url)
    h = url_scaler.transform(pd.DataFrame([extract_url_features(u)]))
    t = url_tfidf.transform([url_for_tfidf(u)])
    x = sparse.hstack([sparse.csr_matrix(h), t]).tocsr()
    return svm_url.predict_proba(x)[0, 1]

sample_urls = df_test['url'].iloc[:200].tolist()
for u in sample_urls[:10]:                               # warm-up
    predict_url_single(u)
times = []
for u in sample_urls:
    t0 = time.perf_counter()
    predict_url_single(u)
    times.append(time.perf_counter() - t0)
url_ms = 1000 * np.median(times)
print(f'\nEnd-to-end latency: {url_ms:.3f} ms/URL (median of {len(times)})')

# --- (e) Log and save ---
log_result('URL', 'B1_single', m_val,
           extra={'split': 'val', 'classifier': f'LinearSVC(C={best_C})', 'latency_ms': url_ms})
joblib.dump(svm_url, CFG['OUT_DIR'] / 'svm_url.joblib')

pd.DataFrame(RESULTS)

## 5.2 HTML Only (Linear SVM)

In [ ]:
# --- (a) Select C on the validation set ---
C_GRID = [0.01, 0.03, 0.1, 0.3, 1, 3, 10]
f1_per_c = {}
for C in C_GRID:
    m = LinearSVC(C=C, max_iter=5000, random_state=SEED).fit(X_html_train, y_train)
    f1_per_c[C] = f1_score(y_val, m.predict(X_html_val))
best_C_html = max(f1_per_c, key=f1_per_c.get)
print('Validation F1 per C:', {c: round(v, 4) for c, v in f1_per_c.items()})
print(f'Selected C: {best_C_html}')

# --- (b) Final model + probability calibration (5-fold CV on TRAIN) ---
eff_html_train = {}
with measure(eff_html_train):
    svm_html = CalibratedClassifierCV(
        LinearSVC(C=best_C_html, max_iter=5000, random_state=SEED),
        method=CFG['CALIB_METHOD'], cv=5).fit(X_html_train, y_train)
print(f'Training time: {eff_html_train["time_s"]:.2f} s')

# --- (c) Validation results ---
P_val['html']  = svm_html.predict_proba(X_html_val)[:, 1]
P_test['html'] = svm_html.predict_proba(X_html_test)[:, 1]

m_val = evaluate(y_val, P_val['html'])
print('\nHTML only (SVM), validation:', {k: round(v, 4) for k, v in m_val.items()})
print('Confusion matrix (rows = true, columns = predicted) [benign, phishing]:')
print(confusion_matrix(y_val, (P_val['html'] >= 0.5).astype(int)))

# --- (d) End-to-end latency per sample (batch size 1) ---
#     raw html_brand_info -> parse nine fields -> structural features -> TF-IDF -> probability
def predict_html_single(html_text: str) -> float:
    row = {f'html_{k}': v for k, v in parse_brand_info(html_text).items()}
    h = html_scaler.transform(pd.DataFrame([extract_html_features(row)]))
    t = html_tfidf.transform([html_for_tfidf(row)])
    x = sparse.hstack([sparse.csr_matrix(h), t]).tocsr()
    return svm_html.predict_proba(x)[0, 1]

sample_html = df_test['html_text'].iloc[:200].tolist()
for s in sample_html[:10]:                               # warm-up
    predict_html_single(s)
times = []
for s in sample_html:
    t0 = time.perf_counter()
    predict_html_single(s)
    times.append(time.perf_counter() - t0)
html_ms = 1000 * np.median(times)
print(f'\nEnd-to-end latency: {html_ms:.3f} ms/sample (median of {len(times)})')

# --- (e) Log and save ---
log_result('HTML', 'B1_single', m_val,
           extra={'split': 'val', 'classifier': f'LinearSVC(C={best_C_html})', 'latency_ms': html_ms})
joblib.dump(svm_html, CFG['OUT_DIR'] / 'svm_html.joblib')

pd.DataFrame(RESULTS)

## 5.3 Image Only (frozen MobileNetV3-Small + RBF SVM)

In [ ]:
# --- (a) Feature extractor: MobileNetV3-Small pretrained on ImageNet, frozen ---
#     include_preprocessing=True, so the input is given directly in the 0-255 range
mbv3 = MobileNetV3Small(input_shape=(*IMG_SIZE_MB, 3), include_top=False,
                        weights='imagenet', pooling='avg', include_preprocessing=True)
mbv3.trainable = False
n_params_mb = mbv3.count_params()
print(f'MobileNetV3-Small: {n_params_mb:,} parameters (~{n_params_mb * 4 / 1e6:.1f} MB) '
      f'| embedding of {mbv3.output_shape[-1]} dimensions')

def embed_mb(X, batch=64):
    ds = tf.data.Dataset.from_tensor_slices(X).map(lambda x: tf.cast(x, tf.float32)).batch(batch)
    return mbv3.predict(ds, verbose=0)

E_img_train = embed_mb(X224_train)
E_img_val   = embed_mb(X224_val)
E_img_test  = embed_mb(X224_test)

# --- (b) Standardize the embeddings ---
img_scaler = StandardScaler()
Z_img_train = img_scaler.fit_transform(E_img_train)
Z_img_val   = img_scaler.transform(E_img_val)
Z_img_test  = img_scaler.transform(E_img_test)

# --- (c) Select C and gamma on the validation set ---
C_GRID     = [0.1, 0.3, 1, 3, 10]
GAMMA_GRID = ['scale', 0.0001, 0.001, 0.01]
grid = {}
for C in C_GRID:
    for g in GAMMA_GRID:
        m = SVC(C=C, gamma=g, kernel='rbf', random_state=SEED).fit(Z_img_train, y_train)
        grid[(C, g)] = f1_score(y_val, m.predict(Z_img_val))
best_C_img, best_gamma_img = max(grid, key=grid.get)
print('Validation F1 (C, gamma):')
print(pd.Series(grid).unstack().round(4))
print(f'Selected: C={best_C_img}, gamma={best_gamma_img}')

# --- (d) Final model + probability calibration (5-fold CV on TRAIN) ---
eff_img_train = {}
with measure(eff_img_train):
    svm_img = CalibratedClassifierCV(
        SVC(C=best_C_img, gamma=best_gamma_img, kernel='rbf', random_state=SEED),
        method=CFG['CALIB_METHOD'], cv=5).fit(Z_img_train, y_train)
print(f'Training time: {eff_img_train["time_s"]:.2f} s')

# --- (e) Validation results ---
P_val['img']  = svm_img.predict_proba(Z_img_val)[:, 1]
P_test['img'] = svm_img.predict_proba(Z_img_test)[:, 1]

m_val = evaluate(y_val, P_val['img'])
print('\nImage only (MobileNetV3-Small + SVM), validation:', {k: round(v, 4) for k, v in m_val.items()})
print('Confusion matrix (rows = true, columns = predicted) [benign, phishing]:')
print(confusion_matrix(y_val, (P_val['img'] >= 0.5).astype(int)))

# --- (f) End-to-end latency per image (batch size 1) ---
#     screenshot file -> viewport crop + resize -> MobileNetV3 -> scaler -> SVM
def predict_img_single(path: str) -> float:
    x = preprocess_image(path, size=IMG_SIZE_MB)[None].astype(np.float32)
    e = mbv3(x, training=False).numpy()
    return svm_img.predict_proba(img_scaler.transform(e))[0, 1]

sample_paths = df_test['screenshot_path'].iloc[:100].tolist()
for p in sample_paths[:5]:                                   # warm-up
    predict_img_single(p)
times = []
for p in sample_paths:
    t0 = time.perf_counter()
    predict_img_single(p)
    times.append(time.perf_counter() - t0)
img_ms = 1000 * np.median(times)
device = 'GPU' if tf.config.list_physical_devices('GPU') else 'CPU'
print(f'\nEnd-to-end latency: {img_ms:.2f} ms/image ({device}, median of {len(times)})')

# --- (g) Log and save ---
log_result('Image', 'B1_single', m_val,
           extra={'split': 'val', 'extractor': 'MobileNetV3-Small (frozen)',
                  'classifier': f'SVC-RBF(C={best_C_img}, gamma={best_gamma_img})',
                  'latency_ms': img_ms})
joblib.dump({'scaler': img_scaler, 'svm': svm_img}, CFG['OUT_DIR'] / 'svm_img.joblib')
np.save(CFG['OUT_DIR'] / 'emb_mbv3_train.npy', E_img_train)
np.save(CFG['OUT_DIR'] / 'emb_mbv3_val.npy',   E_img_val)
np.save(CFG['OUT_DIR'] / 'emb_mbv3_test.npy',  E_img_test)

pd.DataFrame(RESULTS)

## 5.4 Modality Ablation (equal-weight average fusion)

In [ ]:
MODS    = ['url', 'html', 'img']
NAMES   = {'url': 'URL', 'html': 'HTML', 'img': 'Image'}
LATENCY = {'url': url_ms, 'html': html_ms, 'img': img_ms}   # ms/sample from Sections 5.1 to 5.3

def fuse_average(P: dict, mods: tuple) -> np.ndarray:
    """Late fusion: equal-weight average of the phishing probabilities."""
    return np.mean([P[m] for m in mods], axis=0)

# --- (a) All multi-modal combinations ---
for k in (2, 3):
    for mods in combinations(MODS, k):
        name = ' + '.join(NAMES[m] for m in mods)
        P_val[mods]  = fuse_average(P_val, mods)
        P_test[mods] = fuse_average(P_test, mods)
        m_val = evaluate(y_val, P_val[mods])
        log_result(name, 'B1_fusion_avg', m_val,
                   extra={'split': 'val', 'classifier': 'Average probability',
                          # modalities run sequentially, so the latency is the sum of their costs
                          'latency_ms': sum(LATENCY[m] for m in mods)})

# --- (b) Ablation table (validation) ---
tbl1 = (pd.DataFrame(RESULTS)
          .query("group in ['B1_single', 'B1_fusion_avg'] and split == 'val'")
          [['config', 'accuracy', 'precision', 'recall', 'f1', 'fpr', 'auc', 'latency_ms']]
          .drop_duplicates('config', keep='last')
          .reset_index(drop=True))
print('Modality ablation (validation)')
print(tbl1.round(4).to_string(index=False))
tbl1.to_csv(CFG['OUT_DIR'] / 'ablation_val.csv', index=False)

# --- (c) F1 vs latency ---
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(tbl1['latency_ms'], tbl1['f1'], s=60)
for _, r in tbl1.iterrows():
    ax.annotate(r['config'], (r['latency_ms'], r['f1']),
                textcoords='offset points', xytext=(6, 4), fontsize=9)
ax.set_xscale('log')
ax.set_xlabel('Latency per sample (ms, log scale)')
ax.set_ylabel('F1-score (validation)')
ax.set_title('Modality ablation: performance vs cost')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / 'ablation_f1_vs_latency.png', dpi=150)
plt.show()

# 6. Static Fusion Baselines

All baselines use the same modality-specific classifiers and all three modalities.

## 6.1 Early Fusion (feature-level concatenation)

In [ ]:
SVD_DIM = CFG['SVD_DIM']

# --- (a) Reduce the TF-IDF blocks with SVD ---
#     URL TF-IDF (~12k) and HTML TF-IDF (~50k) dimensions vs a 576-d image embedding.
#     Without reduction, the image features would be drowned out by the text features.
svd_url  = TruncatedSVD(n_components=SVD_DIM, random_state=SEED).fit(T_url_train)
svd_html = TruncatedSVD(n_components=SVD_DIM, random_state=SEED).fit(T_html_train)
print(f'Variance retained by SVD, URL: {svd_url.explained_variance_ratio_.sum():.1%} '
      f'| HTML: {svd_html.explained_variance_ratio_.sum():.1%}')

def build_blocks(T_url, H_url_s, T_html, H_html_s, Z_img):
    """Build the three dense modality blocks."""
    return [
        np.hstack([svd_url.transform(T_url),   H_url_s]),     # URL : SVD(TF-IDF) + handcrafted
        np.hstack([svd_html.transform(T_html), H_html_s]),    # HTML: SVD(TF-IDF) + structural
        Z_img,                                                # Image: MobileNetV3 embedding
    ]

B_train = build_blocks(T_url_train, H_url_train_s, T_html_train, H_html_train_s, Z_img_train)
B_val   = build_blocks(T_url_val,   H_url_val_s,   T_html_val,   H_html_val_s,   Z_img_val)
B_test  = build_blocks(T_url_test,  H_url_test_s,  T_html_test,  H_html_test_s,  Z_img_test)

# --- (b) Standardize each block and normalize by sqrt(dimension) ---
#     so that every modality contributes equally regardless of its dimensionality
block_scalers = [StandardScaler().fit(b) for b in B_train]

def concat_blocks(blocks):
    return np.hstack([s.transform(b) / np.sqrt(b.shape[1]) for s, b in zip(block_scalers, blocks)])

X_early_train = concat_blocks(B_train)
X_early_val   = concat_blocks(B_val)
X_early_test  = concat_blocks(B_test)
dims = [b.shape[1] for b in B_train]
print(f'Early fusion dimension: {X_early_train.shape[1]} (URL {dims[0]} + HTML {dims[1]} + Image {dims[2]})')

# --- (c) Select the classifier and hyperparameters on the validation set ---
cand = {}
for C in [0.01, 0.03, 0.1, 0.3, 1, 3]:
    m = LinearSVC(C=C, max_iter=10000, random_state=SEED).fit(X_early_train, y_train)
    cand[('linear', C, None)] = f1_score(y_val, m.predict(X_early_val))
for C in [0.3, 1, 3, 10]:
    for g in ['scale', 0.3, 1, 3]:
        m = SVC(C=C, gamma=g, kernel='rbf', random_state=SEED).fit(X_early_train, y_train)
        cand[('rbf', C, g)] = f1_score(y_val, m.predict(X_early_val))

best_kernel, best_C_early, best_g_early = max(cand, key=cand.get)
top = sorted(cand.items(), key=lambda kv: -kv[1])[:5]
print('Top-5 configurations (validation F1):')
for (k, C, g), f in top:
    print(f'  {k:6} C={C:<5} gamma={g}  ->  {f:.4f}')

# --- (d) Final model + probability calibration (5-fold CV on TRAIN) ---
if best_kernel == 'linear':
    base = LinearSVC(C=best_C_early, max_iter=10000, random_state=SEED)
    clf_desc = f'LinearSVC(C={best_C_early})'
else:
    base = SVC(C=best_C_early, gamma=best_g_early, kernel='rbf', random_state=SEED)
    clf_desc = f'SVC-RBF(C={best_C_early}, gamma={best_g_early})'

eff_early_train = {}
with measure(eff_early_train):
    svm_early = CalibratedClassifierCV(base, method=CFG['CALIB_METHOD'], cv=5).fit(X_early_train, y_train)
print(f'\nSelected classifier: {clf_desc} | training time {eff_early_train["time_s"]:.2f} s')

# --- (e) Validation results ---
P_val['early']  = svm_early.predict_proba(X_early_val)[:, 1]
P_test['early'] = svm_early.predict_proba(X_early_test)[:, 1]

m_val = evaluate(y_val, P_val['early'])
print('Early fusion, validation:', {k: round(v, 4) for k, v in m_val.items()})
print('Confusion matrix (rows = true, columns = predicted) [benign, phishing]:')
print(confusion_matrix(y_val, (P_val['early'] >= 0.5).astype(int)))

# --- (f) End-to-end latency per sample (batch size 1) ---
#     Early fusion ALWAYS needs all three modalities
def predict_early_single(url: str, html_text: str, path: str) -> float:
    # URL
    u = clean_url(url)
    hu = url_scaler.transform(pd.DataFrame([extract_url_features(u)]))
    tu = url_tfidf.transform([url_for_tfidf(u)])
    # HTML
    row = {f'html_{k}': v for k, v in parse_brand_info(html_text).items()}
    hh = html_scaler.transform(pd.DataFrame([extract_html_features(row)]))
    th = html_tfidf.transform([html_for_tfidf(row)])
    # Image
    x = preprocess_image(path, size=IMG_SIZE_MB)[None].astype(np.float32)
    zi = img_scaler.transform(mbv3(x, training=False).numpy())
    # Concatenate
    xe = concat_blocks(build_blocks(tu, hu, th, hh, zi))
    return svm_early.predict_proba(xe)[0, 1]

sample = df_test.iloc[:100]
for _, r in sample.iloc[:5].iterrows():                       # warm-up
    predict_early_single(r['url'], r['html_text'], r['screenshot_path'])
times = []
for _, r in sample.iterrows():
    t0 = time.perf_counter()
    predict_early_single(r['url'], r['html_text'], r['screenshot_path'])
    times.append(time.perf_counter() - t0)
early_ms = 1000 * np.median(times)
print(f'\nEnd-to-end latency: {early_ms:.2f} ms/sample (median of {len(times)})')

# --- (g) Log and save ---
log_result('Early Fusion (Concat)', 'B2_fusion', m_val,
           extra={'split': 'val', 'level': 'Feature-level', 'classifier': clf_desc,
                  'latency_ms': early_ms})
joblib.dump({'svd_url': svd_url, 'svd_html': svd_html, 'block_scalers': block_scalers,
             'svm': svm_early}, CFG['OUT_DIR'] / 'early_fusion.joblib')

pd.DataFrame(RESULTS)[['config', 'group', 'accuracy', 'precision', 'recall', 'f1', 'fpr', 'latency_ms']].round(4)

## 6.2 Majority Voting

In [ ]:
MODS_3 = ['url', 'html', 'img']

def majority_vote(P: dict, mods=MODS_3, thr=0.5):
    """Returns (0/1 prediction, score = fraction of phishing votes, vote matrix)."""
    votes = np.column_stack([(P[m] >= thr).astype(int) for m in mods])   # (n, 3)
    score = votes.mean(axis=1)                   # 0, 1/3, 2/3, or 1
    pred  = (votes.sum(axis=1) >= 2).astype(int) # at least 2 of 3 votes
    return pred, score, votes

# --- (a) Validation and test ---
pred_mv_val, P_val['mv'], votes_val = majority_vote(P_val)
_,           P_test['mv'], _        = majority_vote(P_test)

# evaluate() applies a 0.5 threshold to the score, which equals ">= 2 of 3 votes".
# The score only has four values, so its AUC is coarse.
m_val = evaluate(y_val, P_val['mv'])
print('Majority voting, validation:', {k: round(v, 4) for k, v in m_val.items()})
print('Confusion matrix (rows = true, columns = predicted) [benign, phishing]:')
print(confusion_matrix(y_val, pred_mv_val))

# --- (b) Agreement among modalities ---
n_phish_votes = votes_val.sum(axis=1)
agree = pd.DataFrame({'n_phishing_votes': n_phish_votes, 'correct': (pred_mv_val == y_val)})
summary = agree.groupby('n_phishing_votes').agg(n_samples=('correct', 'size'), accuracy=('correct', 'mean'))
summary['fraction'] = summary['n_samples'] / len(y_val)
print('\nAgreement among modalities (validation):')
print(summary.round(3))
unanimous = np.isin(n_phish_votes, [0, 3]).mean()
print(f'All three modalities agree (0 or 3 votes): {unanimous:.1%} of samples')

# --- (c) Latency: voting needs all three predictions (the vote itself is < 0.01 ms) ---
mv_ms = url_ms + html_ms + img_ms
print(f'\nLatency (sequential): {mv_ms:.2f} ms/sample')

# --- (d) Log ---
log_result('Majority Voting', 'B2_fusion', m_val,
           extra={'split': 'val', 'level': 'Decision-level (late)',
                  'classifier': 'Vote >= 2/3', 'latency_ms': mv_ms})

pd.DataFrame(RESULTS)[['config', 'group', 'accuracy', 'precision', 'recall', 'f1', 'fpr', 'latency_ms']].round(4)

## 6.3 Average Probability and Weighted Average

In [ ]:
def weighted_avg(P: dict, w: dict) -> np.ndarray:
    return sum(w[m] * P[m] for m in MODS_3) / sum(w.values())

# ------------------------------------------------------------
# Average probability: equal weights (same as URL + HTML + Image in Section 5.4)
# ------------------------------------------------------------
w_equal = {m: 1 / 3 for m in MODS_3}
P_val['avg']  = weighted_avg(P_val,  w_equal)
P_test['avg'] = weighted_avg(P_test, w_equal)
m_avg = evaluate(y_val, P_val['avg'])
print('Average probability, validation:', {k: round(v, 4) for k, v in m_avg.items()})

log_result('Average Probability', 'B2_fusion', m_avg,
           extra={'split': 'val', 'level': 'Decision-level (late)', 'classifier': 'w = 1/3 each',
                  'latency_ms': sum(LATENCY.values())})

# ------------------------------------------------------------
# Weighted average: FIXED weights for all samples
# ------------------------------------------------------------
# --- (a) Reference: weights proportional to the F1-score of each modality ---
f1_single = {m: evaluate(y_val, P_val[m])['f1'] for m in MODS_3}
w_f1 = {m: f1_single[m] / sum(f1_single.values()) for m in MODS_3}
m_wf1 = evaluate(y_val, weighted_avg(P_val, w_f1))
print('\nWeights proportional to F1:', {m: round(w, 3) for m, w in w_f1.items()}, '-> F1', round(m_wf1['f1'], 4))

# --- (b) Optimal weights: grid search on the simplex (step 0.05) on the validation set ---
#     w_url + w_html + w_img = 1, each >= 0
STEP = 0.05
ticks = np.round(np.arange(0, 1 + 1e-9, STEP), 2)
grid = []
for wu, wh in product(ticks, ticks):
    wi = round(1 - wu - wh, 2)
    if wi < -1e-9:
        continue
    w = {'url': wu, 'html': wh, 'img': max(wi, 0.0)}
    p = weighted_avg(P_val, w)
    grid.append({**w, 'f1': f1_score(y_val, (p >= 0.5).astype(int)),
                 'logloss': -np.mean(y_val * np.log(p + 1e-12) + (1 - y_val) * np.log(1 - p + 1e-12))})
grid = pd.DataFrame(grid)

# Highest F1; ties are broken by the lowest log-loss (better probabilities)
best = grid.sort_values(['f1', 'logloss'], ascending=[False, True]).iloc[0]
w_opt = {m: float(best[m]) for m in MODS_3}
print('Optimal weights:', w_opt, '-> F1', round(best['f1'], 4))
print('\nTop-5 weight combinations:')
print(grid.sort_values(['f1', 'logloss'], ascending=[False, True]).head().round(4).to_string(index=False))

P_val['wavg']  = weighted_avg(P_val,  w_opt)
P_test['wavg'] = weighted_avg(P_test, w_opt)    # weights from validation are reused as is

m_wavg = evaluate(y_val, P_val['wavg'])
print('\nWeighted average, validation:', {k: round(v, 4) for k, v in m_wavg.items()})
print('Confusion matrix (rows = true, columns = predicted) [benign, phishing]:')
print(confusion_matrix(y_val, (P_val['wavg'] >= 0.5).astype(int)))

# --- (c) F1 for every weight combination ---
piv = grid.pivot(index='html', columns='url', values='f1')
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(piv.sort_index(ascending=False), cmap='viridis', ax=ax,
            cbar_kws={'label': 'F1 (validation)'})
ax.set_xlabel('w_url'); ax.set_ylabel('w_html')
ax.set_title('Weighted average: F1 per weight combination\n(w_img = 1 - w_url - w_html)')
plt.tight_layout()
plt.savefig(FIG_DIR / 'weighted_avg_grid.png', dpi=150)
plt.show()

# --- (d) Latency: modalities with zero weight are skipped ---
wavg_ms = sum(LATENCY[m] for m in MODS_3 if w_opt[m] > 0)
print(f'Latency: {wavg_ms:.2f} ms/sample')

# --- (e) Log ---
log_result('Weighted Average', 'B2_fusion', m_wavg,
           extra={'split': 'val', 'level': 'Decision-level (late)',
                  'classifier': f"w = url {w_opt['url']}, html {w_opt['html']}, img {w_opt['img']}",
                  'latency_ms': wavg_ms})

pd.DataFrame(RESULTS)[['config', 'group', 'accuracy', 'precision', 'recall', 'f1', 'fpr', 'latency_ms']].round(4)

## 6.4 Stacking

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

# --- (a) Out-of-fold (OOF) probabilities on TRAIN ---
#     The meta-classifier must not be trained on probabilities from models that have
#     already seen those samples (they would be over-confident). Each training sample
#     is therefore predicted by a model trained WITHOUT it.
base_models = {
    'url':  (CalibratedClassifierCV(LinearSVC(C=best_C, max_iter=5000, random_state=SEED),
                                    method=CFG['CALIB_METHOD'], cv=5), X_url_train),
    'html': (CalibratedClassifierCV(LinearSVC(C=best_C_html, max_iter=5000, random_state=SEED),
                                    method=CFG['CALIB_METHOD'], cv=5), X_html_train),
    'img':  (CalibratedClassifierCV(SVC(C=best_C_img, gamma=best_gamma_img, kernel='rbf', random_state=SEED),
                                    method=CFG['CALIB_METHOD'], cv=5), Z_img_train),
}

P_oof = {}
for m, (est, X) in base_models.items():
    P_oof[m] = cross_val_predict(est, X, y_train, cv=skf, method='predict_proba', n_jobs=-1)[:, 1]
    print(f'OOF {m:4}: F1 {evaluate(y_train, P_oof[m])["f1"]:.4f}')

# --- (b) Meta features: logits of the probabilities ---
def to_logit(p, eps=1e-6):
    p = np.clip(p, eps, 1 - eps)
    return np.log(p / (1 - p))

def meta_features(P: dict) -> np.ndarray:
    return np.column_stack([to_logit(P[m]) for m in MODS_3])

M_train = meta_features(P_oof)      # OOF (train)
M_val   = meta_features(P_val)      # final models trained on the full training set
M_test  = meta_features(P_test)

# --- (c) Select C of the meta-classifier on the validation set ---
f1_per_c = {}
for C in [0.001, 0.01, 0.1, 1, 10, 100]:
    lr = LogisticRegression(C=C, max_iter=1000).fit(M_train, y_train)
    f1_per_c[C] = f1_score(y_val, lr.predict(M_val))
best_C_meta = max(f1_per_c, key=f1_per_c.get)
print('\nValidation F1 per C (meta):', {c: round(v, 4) for c, v in f1_per_c.items()})
print(f'Selected C: {best_C_meta}')

meta_lr = LogisticRegression(C=best_C_meta, max_iter=1000).fit(M_train, y_train)

coef = pd.Series(meta_lr.coef_[0], index=[NAMES[m] for m in MODS_3])
print('\nMeta-classifier coefficients (larger = more trusted):')
print(coef.round(4).to_string())
print(f'Intercept: {meta_lr.intercept_[0]:.4f}')

# --- (d) Validation results ---
P_val['stack']  = meta_lr.predict_proba(M_val)[:, 1]
P_test['stack'] = meta_lr.predict_proba(M_test)[:, 1]

m_val = evaluate(y_val, P_val['stack'])
print('\nStacking, validation:', {k: round(v, 4) for k, v in m_val.items()})
print('Confusion matrix (rows = true, columns = predicted) [benign, phishing]:')
print(confusion_matrix(y_val, (P_val['stack'] >= 0.5).astype(int)))

# --- (e) Latency: needs all three modalities (meta-classifier < 0.1 ms) ---
stack_ms = url_ms + html_ms + img_ms
print(f'\nLatency (sequential): {stack_ms:.2f} ms/sample')

# --- (f) Log and save ---
log_result('Stacking', 'B2_fusion', m_val,
           extra={'split': 'val', 'level': 'Decision-level (late)',
                  'classifier': f'LogReg(C={best_C_meta}) on OOF logits', 'latency_ms': stack_ms})
joblib.dump(meta_lr, CFG['OUT_DIR'] / 'stacking_meta_lr.joblib')
np.save(CFG['OUT_DIR'] / 'oof_train_probs.npy', np.column_stack([P_oof[m] for m in MODS_3]))

pd.DataFrame(RESULTS)[['config', 'group', 'accuracy', 'precision', 'recall', 'f1', 'fpr', 'latency_ms']].round(4)

## 6.5 Hybrid Static Fusion (early + late)

In [ ]:
SRC = ['url', 'html', 'img', 'early']
SRC_NAMES = {'url': 'URL', 'html': 'HTML', 'img': 'Image', 'early': 'Early fusion'}

def hybrid_avg(P: dict, w: dict) -> np.ndarray:
    return sum(w[s] * P[s] for s in SRC) / sum(w.values())

# --- (a) Reference: weights proportional to F1 ---
f1_src = {s: evaluate(y_val, P_val[s])['f1'] for s in SRC}
w_f1 = {s: f1_src[s] / sum(f1_src.values()) for s in SRC}
m_hf1 = evaluate(y_val, hybrid_avg(P_val, w_f1))
print('F1 per source:', {SRC_NAMES[s]: round(v, 4) for s, v in f1_src.items()})
print('Weights proportional to F1:', {SRC_NAMES[s]: round(w, 3) for s, w in w_f1.items()}, '-> F1', round(m_hf1['f1'], 4))

# --- (b) Optimal weights: grid search on the 4-source simplex (step 0.05) ---
STEP = 0.05
ticks = np.round(np.arange(0, 1 + 1e-9, STEP), 2)
rows = []
for wu, wh, wi in product(ticks, ticks, ticks):
    we = round(1 - wu - wh - wi, 2)
    if we < -1e-9:
        continue
    w = {'url': wu, 'html': wh, 'img': wi, 'early': max(we, 0.0)}
    p = hybrid_avg(P_val, w)
    rows.append({**w, 'f1': f1_score(y_val, (p >= 0.5).astype(int)),
                 'logloss': -np.mean(y_val * np.log(p + 1e-12) + (1 - y_val) * np.log(1 - p + 1e-12))})
grid_h = pd.DataFrame(rows).sort_values(['f1', 'logloss'], ascending=[False, True])
best = grid_h.iloc[0]
w_hyb = {s: float(best[s]) for s in SRC}
print(f'\nTried {len(grid_h)} weight combinations')
print('Optimal weights:', {SRC_NAMES[s]: w for s, w in w_hyb.items()}, '-> F1', round(best['f1'], 4))
print('\nTop-5 weight combinations:')
print(grid_h.head().round(4).to_string(index=False))

# --- (c) Validation results ---
P_val['hybrid']  = hybrid_avg(P_val,  w_hyb)
P_test['hybrid'] = hybrid_avg(P_test, w_hyb)

m_val = evaluate(y_val, P_val['hybrid'])
print('\nHybrid static, validation:', {k: round(v, 4) for k, v in m_val.items()})
print('Confusion matrix (rows = true, columns = predicted) [benign, phishing]:')
print(confusion_matrix(y_val, (P_val['hybrid'] >= 0.5).astype(int)))

# --- (d) End-to-end latency per sample (batch size 1) ---
#     Features of each modality are extracted ONCE and shared by the four classifiers
def predict_hybrid_single(url: str, html_text: str, path: str) -> float:
    u  = clean_url(url)
    hu = url_scaler.transform(pd.DataFrame([extract_url_features(u)]))
    tu = url_tfidf.transform([url_for_tfidf(u)])
    row = {f'html_{k}': v for k, v in parse_brand_info(html_text).items()}
    hh = html_scaler.transform(pd.DataFrame([extract_html_features(row)]))
    th = html_tfidf.transform([html_for_tfidf(row)])
    x  = preprocess_image(path, size=IMG_SIZE_MB)[None].astype(np.float32)
    zi = img_scaler.transform(mbv3(x, training=False).numpy())
    # Path 1: per-modality classifiers
    p = {
        'url':  svm_url.predict_proba(sparse.hstack([sparse.csr_matrix(hu), tu]).tocsr())[0, 1],
        'html': svm_html.predict_proba(sparse.hstack([sparse.csr_matrix(hh), th]).tocsr())[0, 1],
        'img':  svm_img.predict_proba(zi)[0, 1],
    }
    # Path 2: early fusion
    p['early'] = svm_early.predict_proba(concat_blocks(build_blocks(tu, hu, th, hh, zi)))[0, 1]
    # Decision level: fixed weighted average
    return sum(w_hyb[s] * p[s] for s in SRC) / sum(w_hyb.values())

sample = df_test.iloc[:100]
for _, r in sample.iloc[:5].iterrows():                       # warm-up
    predict_hybrid_single(r['url'], r['html_text'], r['screenshot_path'])
times = []
for _, r in sample.iterrows():
    t0 = time.perf_counter()
    predict_hybrid_single(r['url'], r['html_text'], r['screenshot_path'])
    times.append(time.perf_counter() - t0)
hybrid_ms = 1000 * np.median(times)
print(f'\nEnd-to-end latency: {hybrid_ms:.2f} ms/sample (median of {len(times)})')

# --- (e) Log ---
log_result('Hybrid Static', 'B2_fusion', m_val,
           extra={'split': 'val', 'level': 'Hybrid (feature + decision)',
                  'classifier': 'w = ' + ', '.join(f'{SRC_NAMES[s]} {w_hyb[s]}' for s in SRC),
                  'latency_ms': hybrid_ms})

pd.DataFrame(RESULTS)[['config', 'group', 'accuracy', 'precision', 'recall', 'f1', 'fpr', 'latency_ms']].round(4)

# 7. Proposed Method

The modalities are evaluated in order of cost: stage 1 uses the URL, stage 2 adds the HTML, and stage 3 adds the screenshot. After stages 1 and 2, a page exits if the fused probability satisfies `p >= tau_phish` or `p <= 1 - tau_legit`.

Two ablations are reported first, followed by the full proposed method.

## 7.1 Cascade with Fixed Weights (ablation)

In [ ]:
STAGES = [['url'], ['url', 'html'], ['url', 'html', 'img']]

# --- (a) Fixed weights: the optimal weights of the weighted average (Section 6.3) ---
#     At each stage, the weights of the available modalities are renormalized.
W_FIXED = dict(w_opt)
if any(sum(W_FIXED[m] for m in st) == 0 for st in STAGES):
    print('Warning: a stage has zero total weight, so equal weights are used for the cascade')
    W_FIXED = {m: 1 / 3 for m in MODS_3}
print('Fixed cascade weights:', W_FIXED)

def stage_prob(P: dict, mods: list, w: dict) -> np.ndarray:
    tot = sum(w[m] for m in mods)
    return sum(w[m] * P[m] for m in mods) / tot

def run_cascade(P: dict, w: dict, tau_phish: float, tau_legit: float):
    """Returns (final probability, exit stage 1/2/3) per sample.
    A sample exits if p >= tau_phish (confident phishing) or p <= 1 - tau_legit (confident benign)."""
    n = len(next(iter(P.values())))
    p_final = np.full(n, np.nan)
    stage   = np.zeros(n, dtype=int)
    active  = np.ones(n, dtype=bool)
    for k, mods in enumerate(STAGES, start=1):
        p = stage_prob(P, mods, w)
        if k < len(STAGES):
            done = active & ((p >= tau_phish) | (p <= 1 - tau_legit))
        else:
            done = active                                   # final stage: decide all remaining
        p_final[done] = p[done]
        stage[done]   = k
        active &= ~done
    return p_final, stage

def expected_latency(stage: np.ndarray) -> float:
    """Average cost: every sample pays for the URL, samples that continue pay for the HTML, and so on."""
    reach2 = (stage >= 2).mean()
    reach3 = (stage >= 3).mean()
    return LATENCY['url'] + reach2 * LATENCY['html'] + reach3 * LATENCY['img']

# --- (b) Asymmetric threshold sweep on the validation set ---
TAU = CFG['TAU_GRID']                                       # 0.700 ... 0.975
sweep = []
for tp in TAU:
    for tl in TAU:
        p, st = run_cascade(P_val, W_FIXED, tp, tl)
        m = evaluate(y_val, p)
        sweep.append({'tau_phish': tp, 'tau_legit': tl, **m,
                      'exit1': (st == 1).mean(), 'exit2': (st == 2).mean(), 'exit3': (st == 3).mean(),
                      'latency_ms': expected_latency(st)})
sweep = pd.DataFrame(sweep)

# Selection rule: among configurations within 0.005 of the best F1, take the lowest latency
TOL = 0.005
cand = sweep[sweep['f1'] >= sweep['f1'].max() - TOL]
best = cand.sort_values(['latency_ms', 'f1'], ascending=[True, False]).iloc[0]
TAU_P1, TAU_L1 = float(best['tau_phish']), float(best['tau_legit'])
print(f'Best F1 in the sweep: {sweep["f1"].max():.4f}')
print(f'Selected: tau_phish={TAU_P1}, tau_legit={TAU_L1} -> F1 {best["f1"]:.4f}, '
      f'latency {best["latency_ms"]:.2f} ms')

# --- (c) Validation results with the selected thresholds ---
P_val['cascade'],  stage_val  = run_cascade(P_val,  W_FIXED, TAU_P1, TAU_L1)
P_test['cascade'], stage_test = run_cascade(P_test, W_FIXED, TAU_P1, TAU_L1)

m_val = evaluate(y_val, P_val['cascade'])
casc_ms = expected_latency(stage_val)
full_ms = sum(LATENCY.values())
print('\nCascade (fixed weights), validation:', {k: round(v, 4) for k, v in m_val.items()})
print('Confusion matrix (rows = true, columns = predicted) [benign, phishing]:')
print(confusion_matrix(y_val, (P_val['cascade'] >= 0.5).astype(int)))

# Exit rate and accuracy per stage
pred_val = (P_val['cascade'] >= 0.5).astype(int)
per_stage = pd.DataFrame({
    'stage':     ['1: URL', '2: URL+HTML', '3: URL+HTML+Image'],
    'exit_rate': [(stage_val == k).mean() for k in (1, 2, 3)],
    'n_samples': [(stage_val == k).sum() for k in (1, 2, 3)],
    'accuracy':  [(pred_val[stage_val == k] == y_val[stage_val == k]).mean() if (stage_val == k).any() else np.nan
                  for k in (1, 2, 3)],
})
print('\nExit per stage (validation):')
print(per_stage.round(3).to_string(index=False))
print(f'\nMean latency: {casc_ms:.2f} ms vs all three modalities {full_ms:.2f} ms '
      f'-> speedup {full_ms / casc_ms:.1f}x')

# --- (d) Trade-off curve F1 vs latency ---
def pareto(df):
    d = df.sort_values('latency_ms')
    return d[d['f1'] >= d['f1'].cummax()]

front = pareto(sweep)
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(sweep['latency_ms'], sweep['f1'], s=12, alpha=0.3, label='All (tau_phish, tau_legit)')
ax.plot(front['latency_ms'], front['f1'], '-o', ms=4, label='Pareto front')
ax.scatter([casc_ms], [m_val['f1']], s=120, marker='*', zorder=5, label='Selected')
ax.axhline(evaluate(y_val, P_val[('url', 'html')])['f1'], ls='--', lw=1, c='gray', label='URL + HTML (avg)')
ax.set_xlabel('Mean latency per sample (ms)'); ax.set_ylabel('F1 (validation)')
ax.set_title('Cascade (fixed weights): F1 vs latency trade-off')
ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / 'cascade_fixed_tradeoff.png', dpi=150)
plt.show()

# --- (e) Log and save ---
log_result('Cascade', 'B2_proposed', m_val,
           extra={'split': 'val', 'level': 'Proposed (ablation)',
                  'classifier': f'tau_phish={TAU_P1}, tau_legit={TAU_L1}, fixed w',
                  'latency_ms': casc_ms,
                  'exit1': (stage_val == 1).mean(), 'exit2': (stage_val == 2).mean(),
                  'exit3': (stage_val == 3).mean()})
sweep.to_csv(CFG['OUT_DIR'] / 'cascade_fixed_sweep_val.csv', index=False)

pd.DataFrame(RESULTS)[['config', 'group', 'accuracy', 'precision', 'recall', 'f1', 'fpr', 'latency_ms']].round(4)

## 7.2 Dynamic Weighting without Cascade (ablation)

Per-sample weights follow Eq. (1) of the paper: `w_m = r_m * c_m^gamma / sum_j r_j * c_j^gamma`.

In [ ]:
# --- (a) Per-sample confidence measures, both in [0, 1] ---
def conf_margin(p):
    """|2p - 1|: 0 when p = 0.5 (uncertain), 1 when p = 0 or 1 (certain)."""
    return np.abs(2 * p - 1)

def conf_entropy(p, eps=1e-12):
    """1 - binary entropy (bits): 0 when p = 0.5, 1 when p = 0 or 1."""
    p = np.clip(p, eps, 1 - eps)
    return 1 + p * np.log2(p) + (1 - p) * np.log2(1 - p)

CONF_FN = {'margin': conf_margin, 'entropy': conf_entropy}

def dynamic_fusion(P: dict, prior: dict, gamma: float, conf: str = 'margin',
                   mods=MODS_3, available: dict = None, eps: float = 1e-6):
    """Returns (fused probability, per-sample weight matrix).
    available[m] is an optional boolean array, missing modalities get zero weight."""
    n = len(P[mods[0]])
    W = np.zeros((n, len(mods)))
    for j, m in enumerate(mods):
        c = CONF_FN[conf](P[m])
        W[:, j] = prior[m] * (c + eps) ** gamma
        if available is not None:
            W[:, j] *= available[m].astype(float)
    tot = W.sum(axis=1, keepdims=True)
    # If every weight is zero, fall back to the prior
    fallback = np.array([prior[m] for m in mods], dtype=float)
    W = np.where(tot > 0, W / np.where(tot > 0, tot, 1), fallback / fallback.sum())
    p = (W * np.column_stack([P[m] for m in mods])).sum(axis=1)
    return p, W

# --- (b) Select prior, confidence type, and gamma on the validation set ---
PRIORS = {
    'equal': {m: 1 / 3 for m in MODS_3},       # no prior knowledge
    'w_opt': dict(w_opt),                      # weighted-average weights (Section 6.3)
}
GAMMAS = [0, 0.5, 1, 2, 4]
rows = []
for pr_name, prior in PRIORS.items():
    if sum(prior.values()) == 0:
        continue
    for conf in CONF_FN:
        for g in GAMMAS:
            p, _ = dynamic_fusion(P_val, prior, g, conf)
            m = evaluate(y_val, p)
            rows.append({'prior': pr_name, 'conf': conf, 'gamma': g, **m,
                         'logloss': -np.mean(y_val * np.log(p + 1e-12) + (1 - y_val) * np.log(1 - p + 1e-12))})
dyn_grid = pd.DataFrame(rows).sort_values(['f1', 'logloss'], ascending=[False, True])
print('Dynamic weighting configurations (validation), top 10:')
print(dyn_grid.head(10).round(4).to_string(index=False))

best = dyn_grid.iloc[0]
DYN_PRIOR, DYN_CONF, DYN_GAMMA = PRIORS[best['prior']], best['conf'], float(best['gamma'])
print(f'\nSelected: prior={best["prior"]}, conf={DYN_CONF}, gamma={DYN_GAMMA}')
if DYN_GAMMA == 0:
    print('Note: gamma = 0 was selected, so dynamic weighting reduces to static weighting.')

# --- (c) Validation results ---
P_val['dynamic'],  W_val  = dynamic_fusion(P_val,  DYN_PRIOR, DYN_GAMMA, DYN_CONF)
P_test['dynamic'], W_test = dynamic_fusion(P_test, DYN_PRIOR, DYN_GAMMA, DYN_CONF)

m_val = evaluate(y_val, P_val['dynamic'])
print('\nDynamic weighting, validation:', {k: round(v, 4) for k, v in m_val.items()})
print('Confusion matrix (rows = true, columns = predicted) [benign, phishing]:')
print(confusion_matrix(y_val, (P_val['dynamic'] >= 0.5).astype(int)))

# --- (d) Analysis of the per-sample weights ---
W_df = pd.DataFrame(W_val, columns=[NAMES[m] for m in MODS_3])
print('\nPer-sample weight statistics (validation):')
print(W_df.describe().T[['mean', 'std', 'min', '50%', 'max']].round(3))

# Does dynamic weighting help in the uncertain zone compared with static weights?
p_uh   = P_val[('url', 'html')]
unsure = (p_uh > 0.3) & (p_uh < 0.7)
for name, p in [('Weighted Avg (static)', P_val['wavg']), ('Dynamic Weighting', P_val['dynamic'])]:
    acc = ((p[unsure] >= 0.5).astype(int) == y_val[unsure]).mean()
    print(f'{name:22} | accuracy in the uncertain URL+HTML zone ({unsure.sum()} samples): {acc:.1%}')

fig, ax = plt.subplots(figsize=(8, 4))
for col in W_df.columns:
    ax.hist(W_df[col], bins=30, alpha=0.5, label=col)
ax.set_xlabel('Per-sample weight'); ax.set_ylabel('Number of samples')
ax.set_title(f'Distribution of dynamic weights (prior={best["prior"]}, conf={DYN_CONF}, gamma={DYN_GAMMA})')
ax.legend(); plt.tight_layout()
plt.savefig(FIG_DIR / 'dynamic_weight_distribution.png', dpi=150)
plt.show()

# --- (e) Latency: without a cascade, all three modalities always run ---
dyn_ms = sum(LATENCY.values())
print(f'Latency: {dyn_ms:.2f} ms/sample')

# --- (f) Log ---
log_result('Dynamic Weighting', 'B2_proposed', m_val,
           extra={'split': 'val', 'level': 'Proposed (ablation)',
                  'classifier': f'prior={best["prior"]}, conf={DYN_CONF}, gamma={DYN_GAMMA}',
                  'latency_ms': dyn_ms})
dyn_grid.to_csv(CFG['OUT_DIR'] / 'dynamic_weighting_grid_val.csv', index=False)

pd.DataFrame(RESULTS)[['config', 'group', 'accuracy', 'precision', 'recall', 'f1', 'fpr', 'latency_ms']].round(4)

## 7.3 Cascade + Dynamic Weighting (proposed method)

In [ ]:
# --- (a) Cascade with dynamic fusion at every stage (Eq. 1 to Eq. 3) ---
def run_cascade_dynamic(P: dict, prior: dict, gamma: float, conf: str,
                        tau_phish: float, tau_legit: float):
    """Returns (final probability, exit stage 1/2/3)."""
    n = len(P['url'])
    p_final = np.full(n, np.nan)
    stage   = np.zeros(n, dtype=int)
    active  = np.ones(n, dtype=bool)
    for k, mods in enumerate(STAGES, start=1):
        p, _ = dynamic_fusion(P, prior, gamma, conf, mods=mods)
        if k < len(STAGES):
            done = active & ((p >= tau_phish) | (p <= 1 - tau_legit))
        else:
            done = active
        p_final[done] = p[done]
        stage[done]   = k
        active &= ~done
    return p_final, stage

# --- (b) Select gamma and the thresholds jointly on the validation set ---
#     The prior and confidence type follow Section 7.2. Gamma is swept again because
#     the best gamma without a cascade is not necessarily the best with a cascade.
rows = []
for g in GAMMAS:
    for tp in TAU:
        for tl in TAU:
            p, st = run_cascade_dynamic(P_val, DYN_PRIOR, g, DYN_CONF, tp, tl)
            m = evaluate(y_val, p)
            rows.append({'gamma': g, 'tau_phish': tp, 'tau_legit': tl, **m,
                         'exit1': (st == 1).mean(), 'exit2': (st == 2).mean(), 'exit3': (st == 3).mean(),
                         'latency_ms': expected_latency(st)})
sweep_cd = pd.DataFrame(rows)

# Same selection rule as Section 7.1
cand = sweep_cd[sweep_cd['f1'] >= sweep_cd['f1'].max() - TOL]
best = cand.sort_values(['latency_ms', 'f1'], ascending=[True, False]).iloc[0]
CD_GAMMA, CD_TP, CD_TL = float(best['gamma']), float(best['tau_phish']), float(best['tau_legit'])
print(f'Tried {len(sweep_cd)} configurations | best F1: {sweep_cd["f1"].max():.4f}')
print(f'Selected: gamma={CD_GAMMA}, tau_phish={CD_TP}, tau_legit={CD_TL} '
      f'-> F1 {best["f1"]:.4f}, latency {best["latency_ms"]:.2f} ms')

# --- (c) Validation results ---
P_val['casc_dyn'],  stage_cd_val  = run_cascade_dynamic(P_val,  DYN_PRIOR, CD_GAMMA, DYN_CONF, CD_TP, CD_TL)
P_test['casc_dyn'], stage_cd_test = run_cascade_dynamic(P_test, DYN_PRIOR, CD_GAMMA, DYN_CONF, CD_TP, CD_TL)

m_val = evaluate(y_val, P_val['casc_dyn'])
cd_ms = expected_latency(stage_cd_val)
print('\nCascade + dynamic weighting, validation:', {k: round(v, 4) for k, v in m_val.items()})
print('Confusion matrix (rows = true, columns = predicted) [benign, phishing]:')
print(confusion_matrix(y_val, (P_val['casc_dyn'] >= 0.5).astype(int)))

pred_cd = (P_val['casc_dyn'] >= 0.5).astype(int)
per_stage = pd.DataFrame({
    'stage':     ['1: URL', '2: URL+HTML', '3: URL+HTML+Image'],
    'exit_rate': [(stage_cd_val == k).mean() for k in (1, 2, 3)],
    'n_samples': [(stage_cd_val == k).sum() for k in (1, 2, 3)],
    'accuracy':  [(pred_cd[stage_cd_val == k] == y_val[stage_cd_val == k]).mean()
                  if (stage_cd_val == k).any() else np.nan for k in (1, 2, 3)],
})
print('\nExit per stage (validation):')
print(per_stage.round(3).to_string(index=False))
print(f'\nMean latency: {cd_ms:.2f} ms vs all three modalities {full_ms:.2f} ms '
      f'-> speedup {full_ms / cd_ms:.1f}x')

# --- (d) Pareto fronts: fixed-weight cascade vs dynamic cascade ---
front_fixed = pareto(sweep)
front_dyn   = pareto(sweep_cd)
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(front_fixed['latency_ms'], front_fixed['f1'], '-o', ms=4, label='Cascade (fixed weights)')
ax.plot(front_dyn['latency_ms'],   front_dyn['f1'],   '-s', ms=4, label='Cascade + dynamic (proposed)')
ax.scatter([cd_ms], [m_val['f1']], s=150, marker='*', zorder=5, label='Proposed (selected)')
ax.axhline(evaluate(y_val, P_val[('url', 'html')])['f1'], ls='--', lw=1, c='gray', label='URL + HTML (avg)')
ax.set_xlabel('Mean latency per sample (ms)'); ax.set_ylabel('F1 (validation)')
ax.set_title('Pareto front: F1 vs latency')
ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / 'cascade_dynamic_vs_fixed_pareto.png', dpi=150)
plt.show()

# --- (e) Log and save ---
log_result('Cascade + Dynamic Weighting', 'B2_proposed', m_val,
           extra={'split': 'val', 'level': 'Proposed',
                  'classifier': f'gamma={CD_GAMMA}, tau_phish={CD_TP}, tau_legit={CD_TL}, conf={DYN_CONF}',
                  'latency_ms': cd_ms,
                  'exit1': (stage_cd_val == 1).mean(), 'exit2': (stage_cd_val == 2).mean(),
                  'exit3': (stage_cd_val == 3).mean()})
sweep_cd.to_csv(CFG['OUT_DIR'] / 'cascade_dynamic_sweep_val.csv', index=False)

# --- (f) Summary of all fusion methods (validation) ---
ORDER = ['URL + HTML', 'Early Fusion (Concat)', 'Majority Voting', 'Average Probability',
         'Weighted Average', 'Stacking', 'Hybrid Static',
         'Cascade', 'Dynamic Weighting', 'Cascade + Dynamic Weighting']
res = pd.DataFrame(RESULTS).drop_duplicates('config', keep='last').set_index('config')
tbl2 = res.reindex(index=[c for c in ORDER if c in res.index],
                   columns=['level', 'accuracy', 'precision', 'recall', 'f1', 'fpr', 'auc', 'latency_ms']).copy()
tbl2.loc['URL + HTML', 'level'] = 'Reference (2 modalities)'
tbl2['speedup_vs_full'] = full_ms / tbl2['latency_ms']
print('\nComparison of fusion methods (validation)')
print(tbl2.round(4).to_string())
tbl2.to_csv(CFG['OUT_DIR'] / 'fusion_comparison_val.csv')

# 8. Final Evaluation on the Test Set

The test set is evaluated **once**. Every hyperparameter, weight, and threshold was fixed on the validation set. The 95% confidence interval of each F1-score and the paired differences against the proposed method are estimated with 1,000 bootstrap resamples (the same indices for every method).

In [ ]:
CONFIGS = {
    # Modality ablation
    'URL':                         ('ablation', 'url'),
    'HTML':                        ('ablation', 'html'),
    'Image':                       ('ablation', 'img'),
    'URL + HTML':                  ('ablation', ('url', 'html')),
    'URL + Image':                 ('ablation', ('url', 'img')),
    'HTML + Image':                ('ablation', ('html', 'img')),
    'URL + HTML + Image':          ('ablation', ('url', 'html', 'img')),
    # Fusion methods (three modalities)
    'Early Fusion (Concat)':       ('fusion', 'early'),
    'Majority Voting':             ('fusion', 'mv'),
    'Average Probability':         ('fusion', 'avg'),
    'Weighted Average':            ('fusion', 'wavg'),
    'Stacking':                    ('fusion', 'stack'),
    'Hybrid Static':               ('fusion', 'hybrid'),
    'Cascade':                     ('fusion', 'cascade'),
    'Dynamic Weighting':           ('fusion', 'dynamic'),
    'Cascade + Dynamic Weighting': ('fusion', 'casc_dyn'),
}

# Latency: static methods reuse the validation value (independent of the data),
# cascades are recomputed from their exit rates on the TEST set.
lat_val = pd.DataFrame(RESULTS).drop_duplicates('config', keep='last').set_index('config')['latency_ms']
lat_test = lat_val.to_dict()
lat_test['Cascade'] = expected_latency(stage_test)
lat_test['Cascade + Dynamic Weighting'] = expected_latency(stage_cd_test)

# --- (a) Bootstrap with the same indices for all methods (paired comparison) ---
N_BOOT = 1000
rng = np.random.default_rng(SEED)
boot_idx = rng.integers(0, len(y_test), size=(N_BOOT, len(y_test)))

def f1_boot(p):
    pred = (p >= 0.5).astype(int)
    return np.array([f1_score(y_test[i], pred[i], zero_division=0) for i in boot_idx])

rows, F1_BOOT = [], {}
for name, (part, key) in CONFIGS.items():
    if key not in P_test:
        print(f'Warning: {name} has no test probabilities and is skipped')
        continue
    p = P_test[key]
    m = evaluate(y_test, p)
    F1_BOOT[name] = f1_boot(p)
    lo, hi = np.percentile(F1_BOOT[name], [2.5, 97.5])
    rows.append({'part': part, 'config': name, **m, 'f1_ci95': f'[{lo:.3f}, {hi:.3f}]',
                 'latency_ms': lat_test.get(name, np.nan)})

final = pd.DataFrame(rows).set_index('config')
full_ms = url_ms + html_ms + img_ms
final['speedup_vs_full'] = full_ms / final['latency_ms']

# Cascade exit rates on the test set
for name, st in [('Cascade', stage_test), ('Cascade + Dynamic Weighting', stage_cd_test)]:
    for k in (1, 2, 3):
        final.loc[name, f'exit{k}'] = (st == k).mean()

pd.set_option('display.width', 200)
print('=== FINAL RESULTS, TEST SET ===\n')
for part, title in [('ablation', 'Modality ablation'), ('fusion', 'Fusion methods')]:
    print(title)
    cols = ['accuracy', 'precision', 'recall', 'f1', 'f1_ci95', 'fpr', 'auc', 'latency_ms', 'speedup_vs_full']
    print(final[final['part'] == part][cols].round(4).to_string(), '\n')

print('Cascade exit rates (test):')
print(final.loc[['Cascade', 'Cascade + Dynamic Weighting'], ['exit1', 'exit2', 'exit3']].round(3), '\n')

# --- (b) Paired test: proposed method vs every other method ---
#     F1 difference (proposed - other) with a 95% bootstrap CI.
#     If the CI does not contain 0, the difference is significant.
PROPOSED = 'Cascade + Dynamic Weighting'
print(f'F1 difference ({PROPOSED} - other), test set:')
diff_rows = []
for name in F1_BOOT:
    if name == PROPOSED:
        continue
    d = F1_BOOT[PROPOSED] - F1_BOOT[name]
    lo, hi = np.percentile(d, [2.5, 97.5])
    diff_rows.append({'compared_with': name,
                      'f1_diff': final.loc[PROPOSED, 'f1'] - final.loc[name, 'f1'],
                      'ci95_low': lo, 'ci95_high': hi,
                      'conclusion': 'better' if lo > 0 else ('worse' if hi < 0 else 'no significant difference')})
diff = pd.DataFrame(diff_rows).set_index('compared_with')
print(diff.round(4).to_string())

# --- (c) Save ---
final.to_csv(CFG['OUT_DIR'] / 'final_test_results.csv')
diff.to_csv(CFG['OUT_DIR'] / 'final_test_paired_diff.csv')

# 9. Efficiency Evaluation (CPU, batch size 1)

Every method is run end to end on raw test samples, one page at a time on the CPU. Modalities are computed lazily, so a cascade only pays for the modalities it actually uses.

In [ ]:
N_EFF   = 200          # number of test samples measured
WARMUP  = 5
CPU_DEV = '/CPU:0'
proc    = psutil.Process()

print(f'CPU: {platform.processor() or platform.machine()} | {os.cpu_count()} logical cores')
print(f'Measured on {N_EFF} test samples, batch size 1, device CPU\n')

# --- (a) Lazy inference: a modality is computed only when it is needed ---
class Sample:
    """Holds one raw sample. Features and probabilities are computed on demand (once)."""
    def __init__(self, row):
        self.row, self._c = row, {}

    def _get(self, key, fn):
        if key not in self._c:
            self._c[key] = fn()
        return self._c[key]

    # Features
    def f_url(self):
        def fn():
            u = clean_url(self.row['url'])
            hu = url_scaler.transform(pd.DataFrame([extract_url_features(u)]))
            tu = url_tfidf.transform([url_for_tfidf(u)])
            return hu, tu
        return self._get('f_url', fn)

    def f_html(self):
        def fn():
            r = {f'html_{k}': v for k, v in parse_brand_info(self.row['html_text']).items()}
            hh = html_scaler.transform(pd.DataFrame([extract_html_features(r)]))
            th = html_tfidf.transform([html_for_tfidf(r)])
            return hh, th
        return self._get('f_html', fn)

    def f_img(self):
        def fn():
            x = preprocess_image(self.row['screenshot_path'], size=IMG_SIZE_MB)[None].astype(np.float32)
            with tf.device(CPU_DEV):
                e = mbv3(x, training=False).numpy()
            return img_scaler.transform(e)
        return self._get('f_img', fn)

    # Probability per modality
    def p(self, m):
        def fn():
            if m == 'url':
                hu, tu = self.f_url()
                return svm_url.predict_proba(sparse.hstack([sparse.csr_matrix(hu), tu]).tocsr())[0, 1]
            if m == 'html':
                hh, th = self.f_html()
                return svm_html.predict_proba(sparse.hstack([sparse.csr_matrix(hh), th]).tocsr())[0, 1]
            if m == 'img':
                return svm_img.predict_proba(self.f_img())[0, 1]
            if m == 'early':
                hu, tu = self.f_url(); hh, th = self.f_html(); zi = self.f_img()
                return svm_early.predict_proba(concat_blocks(build_blocks(tu, hu, th, hh, zi)))[0, 1]
        return self._get(f'p_{m}', fn)

def _dyn(s, mods, prior, gamma, conf):
    P1 = {m: np.array([s.p(m)]) for m in mods}
    return float(dynamic_fusion(P1, prior, gamma, conf, mods=mods)[0][0])

def _cascade(s, fuse, tp, tl):
    for k, mods in enumerate(STAGES, start=1):
        p = fuse(s, mods)
        if k == len(STAGES) or p >= tp or p <= 1 - tl:
            return p, k

def _wfix(s, mods):
    tot = sum(W_FIXED[m] for m in mods)
    return sum(W_FIXED[m] * s.p(m) for m in mods) / tot

# --- (b) Inference of every method, returning (probability, exit stage) ---
METHODS = {
    'URL':                         lambda s: (s.p('url'), None),
    'HTML':                        lambda s: (s.p('html'), None),
    'Image':                       lambda s: (s.p('img'), None),
    'URL + HTML':                  lambda s: ((s.p('url') + s.p('html')) / 2, None),
    'Early Fusion (Concat)':       lambda s: (s.p('early'), None),
    'Majority Voting':             lambda s: (np.mean([s.p(m) >= 0.5 for m in MODS_3]), None),
    'Average Probability':         lambda s: (np.mean([s.p(m) for m in MODS_3]), None),
    'Weighted Average':            lambda s: (sum(w_opt[m] * s.p(m) for m in MODS_3 if w_opt[m] > 0)
                                              / sum(w_opt.values()), None),
    'Stacking':                    lambda s: (meta_lr.predict_proba(
                                              np.array([[to_logit(s.p(m)) for m in MODS_3]]))[0, 1], None),
    'Hybrid Static':               lambda s: (sum(w_hyb[k] * s.p(k) for k in SRC if w_hyb[k] > 0)
                                              / sum(w_hyb.values()), None),
    'Cascade':                     lambda s: _cascade(s, _wfix, TAU_P1, TAU_L1),
    'Dynamic Weighting':           lambda s: (_dyn(s, MODS_3, DYN_PRIOR, DYN_GAMMA, DYN_CONF), None),
    'Cascade + Dynamic Weighting': lambda s: _cascade(
                                              s, lambda s_, mods: _dyn(s_, mods, DYN_PRIOR, CD_GAMMA, DYN_CONF),
                                              CD_TP, CD_TL),
}

# --- (c) Measurement ---
rows_eff = df_test.iloc[:N_EFF]
ref_key = {'URL': 'url', 'HTML': 'html', 'Image': 'img', 'URL + HTML': ('url', 'html'),
           'Early Fusion (Concat)': 'early', 'Majority Voting': 'mv', 'Average Probability': 'avg',
           'Weighted Average': 'wavg', 'Stacking': 'stack', 'Hybrid Static': 'hybrid',
           'Cascade': 'cascade', 'Dynamic Weighting': 'dynamic', 'Cascade + Dynamic Weighting': 'casc_dyn'}

eff_rows = []
for name, fn in METHODS.items():
    for _, r in rows_eff.iloc[:WARMUP].iterrows():               # warm-up
        fn(Sample(r))
    gc.collect()
    rss0 = proc.memory_info().rss
    peak = rss0
    cpu0 = sum(proc.cpu_times()[:2])
    lat, probs, stages = [], [], []
    t_all = time.perf_counter()
    for _, r in tqdm(rows_eff.iterrows(), total=len(rows_eff), desc=name, leave=False):
        t0 = time.perf_counter()
        p, k = fn(Sample(r))
        lat.append(time.perf_counter() - t0)
        probs.append(p); stages.append(k)
        peak = max(peak, proc.memory_info().rss)
    wall = time.perf_counter() - t_all
    cpu_used = sum(proc.cpu_times()[:2]) - cpu0
    lat = np.array(lat) * 1000

    # Sanity check: real per-sample inference must agree with the vectorized P_test
    ref = P_test[ref_key[name]][:N_EFF]
    agree = np.mean((np.array(probs) >= 0.5) == (ref >= 0.5))

    row = {'config': name,
           'latency_median_ms': np.median(lat), 'latency_mean_ms': lat.mean(),
           'latency_p95_ms': np.percentile(lat, 95),
           'throughput_per_s': len(lat) / wall,
           'cpu_ms_per_sample': 1000 * cpu_used / len(lat),
           'cpu_util_%': 100 * cpu_used / wall,
           'ram_peak_delta_mb': (peak - rss0) / 1e6,
           'agree_with_P_test': agree}
    if stages[0] is not None:
        st = np.array(stages)
        row.update({f'exit{k}': (st == k).mean() for k in (1, 2, 3)})
    eff_rows.append(row)

eff = pd.DataFrame(eff_rows).set_index('config')

# --- (d) Model size (MB) per method ---
def obj_mb(*objs):
    buf = io.BytesIO(); joblib.dump(objs, buf); return buf.tell() / 1e6

size_part = {
    'url':   obj_mb(url_scaler, url_tfidf, svm_url),
    'html':  obj_mb(html_scaler, html_tfidf, svm_html),
    'img':   n_params_mb * 4 / 1e6 + obj_mb(img_scaler, svm_img),
    'early': obj_mb(svd_url, svd_html, block_scalers, svm_early),
    'meta':  obj_mb(meta_lr),
}
uses = {
    'URL': ['url'], 'HTML': ['html'], 'Image': ['img'], 'URL + HTML': ['url', 'html'],
    'Early Fusion (Concat)': ['url', 'html', 'img', 'early'],
    'Majority Voting': ['url', 'html', 'img'], 'Average Probability': ['url', 'html', 'img'],
    'Weighted Average': [m for m in MODS_3 if w_opt[m] > 0],
    'Stacking': ['url', 'html', 'img', 'meta'],
    'Hybrid Static': ['url', 'html', 'img'] + (['early'] if w_hyb['early'] > 0 else []),
    'Cascade': ['url', 'html', 'img'], 'Dynamic Weighting': ['url', 'html', 'img'],
    'Cascade + Dynamic Weighting': ['url', 'html', 'img'],
}
eff['model_size_mb'] = [sum(size_part[c] for c in uses[n]) for n in eff.index]

# Merge with the test F1 (Section 8). Speedup is relative to average probability fusion,
# which always processes all three modalities.
full_ref = eff.loc['Average Probability', 'latency_mean_ms']
eff['speedup_vs_full'] = full_ref / eff['latency_mean_ms']
eff['f1_test'] = final['f1'].reindex(eff.index)

cols = ['f1_test', 'latency_median_ms', 'latency_mean_ms', 'latency_p95_ms', 'throughput_per_s',
        'speedup_vs_full', 'cpu_ms_per_sample', 'cpu_util_%', 'ram_peak_delta_mb', 'model_size_mb']
print('=== EFFICIENCY (CPU, batch size 1, test set) ===')
print(eff[cols].round(3).to_string())

print('\nCascade exit rates (real inference):')
print(eff.loc[['Cascade', 'Cascade + Dynamic Weighting'], ['exit1', 'exit2', 'exit3']].round(3))

print('\nSanity check (agreement of real inference with P_test):')
print(eff['agree_with_P_test'].round(4).to_string())
if (eff['agree_with_P_test'] < 0.99).any():
    print('Warning: a method agrees on < 99% of samples, check its inference function.')

print('\nCost per modality (median, ms): '
      f"URL {eff.loc['URL','latency_median_ms']:.2f} | HTML {eff.loc['HTML','latency_median_ms']:.2f} | "
      f"Image {eff.loc['Image','latency_median_ms']:.2f}")

# --- (e) Figures ---
FUSION_ORDER = ['URL + HTML', 'Early Fusion (Concat)', 'Majority Voting', 'Average Probability',
                'Weighted Average', 'Stacking', 'Hybrid Static', 'Cascade', 'Dynamic Weighting',
                'Cascade + Dynamic Weighting']
e2 = eff.loc[[c for c in FUSION_ORDER if c in eff.index]]

fig, axes = plt.subplots(1, 2, figsize=(16, 5.5))
# (1) Test F1 vs mean latency
ax = axes[0]
for name, r in e2.iterrows():
    star = name == 'Cascade + Dynamic Weighting'
    ax.scatter(r['latency_mean_ms'], r['f1_test'], s=160 if star else 50,
               marker='*' if star else 'o', zorder=3)
    ax.annotate(name, (r['latency_mean_ms'], r['f1_test']), textcoords='offset points',
                xytext=(6, 4), fontsize=8)
ax.set_xscale('log')
ax.set_xlabel('Mean latency per sample (ms, CPU, log scale)')
ax.set_ylabel('F1 (test)')
ax.set_title('Performance vs efficiency')
ax.grid(alpha=0.3)

# (2) Mean latency per method
ax = axes[1]
e2s = e2.sort_values('latency_mean_ms')
ax.barh(e2s.index, e2s['latency_mean_ms'])
for i, v in enumerate(e2s['latency_mean_ms']):
    ax.text(v, i, f' {v:.1f} ms', va='center', fontsize=8)
ax.set_xlabel('Mean latency per sample (ms, CPU)')
ax.set_title('Latency per fusion method')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / 'efficiency_cpu.png', dpi=150)
plt.show()

# --- (f) Save ---
eff.to_csv(CFG['OUT_DIR'] / 'efficiency_cpu_test.csv')

# 10. Visualization of the Proposed Method

In [ ]:
PROP_KEY, PROP_NAME = 'casc_dyn', 'Cascade + Dynamic (proposed)'
COMPARE = {                                  # main comparison curves
    PROP_KEY:            PROP_NAME,
    ('url', 'html'):     'URL + HTML (avg)',
    'wavg':              'Weighted Average',
    'stack':             'Stacking',
    'early':             'Early Fusion',
    'avg':               'Average Probability',
}
p_prop    = P_test[PROP_KEY]
pred_prop = (p_prop >= 0.5).astype(int)
st_prop   = stage_cd_test
STAGE_LBL = {1: 'Stage 1\nURL', 2: 'Stage 2\nURL+HTML', 3: 'Stage 3\nURL+HTML+Image'}

def save(fig, name):
    fig.tight_layout()
    fig.savefig(FIG_DIR / f'{name}.png', dpi=200, bbox_inches='tight')
    plt.show()

# ------------------------------------------------------------
# G1. Confusion matrix (counts + row percentages)
# ------------------------------------------------------------
cm = confusion_matrix(y_test, pred_prop)
cm_pct = cm / cm.sum(axis=1, keepdims=True)
annot = np.array([[f'{cm[i, j]}\n({cm_pct[i, j]:.1%})' for j in range(2)] for i in range(2)])
fig, ax = plt.subplots(figsize=(5, 4.3))
sns.heatmap(cm_pct, annot=annot, fmt='', cmap='Blues', vmin=0, vmax=1, cbar=False,
            xticklabels=['Benign', 'Phishing'], yticklabels=['Benign', 'Phishing'], ax=ax)
ax.set_xlabel('Predicted'); ax.set_ylabel('True label')
ax.set_title(f'Confusion matrix, {PROP_NAME}')
save(fig, 'G1_confusion_matrix')

# ------------------------------------------------------------
# G2. ROC and precision-recall curves: proposed vs baselines
# ------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for key, lbl in COMPARE.items():
    if key not in P_test:
        continue
    p = P_test[key]
    lw, ls = (2.5, '-') if key == PROP_KEY else (1.2, '--')
    fpr, tpr, _ = roc_curve(y_test, p)
    axes[0].plot(fpr, tpr, lw=lw, ls=ls, label=f'{lbl} (AUC {sk_auc(fpr, tpr):.3f})')
    prec, rec, _ = precision_recall_curve(y_test, p)
    axes[1].plot(rec, prec, lw=lw, ls=ls, label=f'{lbl} (AP {average_precision_score(y_test, p):.3f})')
axes[0].plot([0, 1], [0, 1], c='gray', lw=0.8)
axes[0].set_xlabel('False positive rate'); axes[0].set_ylabel('True positive rate'); axes[0].set_title('ROC curve')
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision'); axes[1].set_title('Precision-recall curve')
for ax in axes:
    ax.legend(fontsize=8, loc='lower right' if ax is axes[0] else 'lower left'); ax.grid(alpha=0.3)
save(fig, 'G2_roc_pr_curves')

# ------------------------------------------------------------
# G3. Cascade flow: how many samples exit at each stage and how accurately
# ------------------------------------------------------------
stage_tbl = pd.DataFrame({
    'stage':   [STAGE_LBL[k] for k in (1, 2, 3)],
    'correct': [((st_prop == k) & (pred_prop == y_test)).sum() for k in (1, 2, 3)],
    'wrong':   [((st_prop == k) & (pred_prop != y_test)).sum() for k in (1, 2, 3)],
})
stage_tbl['total'] = stage_tbl['correct'] + stage_tbl['wrong']
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(stage_tbl['stage'], stage_tbl['correct'], label='Correct')
ax.bar(stage_tbl['stage'], stage_tbl['wrong'], bottom=stage_tbl['correct'], label='Wrong')
for i, r in stage_tbl.iterrows():
    acc = r['correct'] / r['total'] if r['total'] else np.nan
    ax.text(i, r['total'], f"{r['total'] / len(y_test):.1%} of samples\naccuracy {acc:.1%}",
            ha='center', va='bottom', fontsize=9)
ax.set_ylabel('Number of samples (test)')
ax.set_title('Cascade exit distribution and accuracy per stage')
ax.set_ylim(0, stage_tbl['total'].max() * 1.25); ax.legend()
save(fig, 'G3_cascade_exit_per_stage')

# ------------------------------------------------------------
# G4. Final probability distribution per class with the threshold zones
# ------------------------------------------------------------
fig, ax = plt.subplots(figsize=(9, 4.5))
bins = np.linspace(0, 1, 41)
ax.hist(p_prop[y_test == 0], bins=bins, alpha=0.6, label='Benign')
ax.hist(p_prop[y_test == 1], bins=bins, alpha=0.6, label='Phishing')
ax.axvline(0.5, c='k', lw=1, label='Decision threshold 0.5')
ax.axvline(CD_TP, c='red', ls='--', lw=1, label=f'tau_phish = {CD_TP}')
ax.axvline(1 - CD_TL, c='green', ls='--', lw=1, label=f'1 - tau_legit = {1 - CD_TL:.3f}')
ax.set_xlabel('Phishing probability (final output)'); ax.set_ylabel('Number of samples')
ax.set_title('Class separation of the proposed method'); ax.legend(fontsize=8)
save(fig, 'G4_probability_distribution')

# ------------------------------------------------------------
# G5. Dynamic weights of the samples that reach stage 3
#     (shows when the screenshot actually influences the decision)
# ------------------------------------------------------------
_, W3 = dynamic_fusion(P_test, DYN_PRIOR, CD_GAMMA, DYN_CONF, mods=MODS_3)
mask3 = st_prop == 3
if mask3.sum() > 0:
    Wd = pd.DataFrame(W3[mask3], columns=[NAMES[m] for m in MODS_3])
    Wd['result'] = np.where(pred_prop[mask3] == y_test[mask3], 'Correct', 'Wrong')
    Wl = Wd.melt(id_vars='result', var_name='modality', value_name='weight')
    fig, ax = plt.subplots(figsize=(8, 4.5))
    sns.boxplot(data=Wl, x='modality', y='weight', hue='result', ax=ax)
    ax.set_title(f'Per-sample dynamic weights at stage 3 (n = {mask3.sum()})')
    ax.set_ylabel('Weight (per sample)'); ax.set_xlabel('')
    save(fig, 'G5_dynamic_weights_stage3')
else:
    print('G5 skipped: no sample reaches stage 3.')

# ------------------------------------------------------------
# G6. Reliability diagram: the cascade relies on calibrated probabilities to exit
# ------------------------------------------------------------
fig, ax = plt.subplots(figsize=(6, 5.5))
ax.plot([0, 1], [0, 1], c='gray', lw=0.8, label='Perfect calibration')
for key in [PROP_KEY, ('url', 'html'), 'stack', 'wavg']:
    if key not in P_test:
        continue
    frac, mean_p = calibration_curve(y_test, P_test[key], n_bins=10, strategy='quantile')
    ax.plot(mean_p, frac, marker='o', lw=2.5 if key == PROP_KEY else 1.2, label=COMPARE[key])
ax.set_xlabel('Mean predicted probability (per bin)'); ax.set_ylabel('Observed fraction of phishing')
ax.set_title('Reliability diagram (test)'); ax.legend(fontsize=8); ax.grid(alpha=0.3)
save(fig, 'G6_reliability_diagram')

# ------------------------------------------------------------
# G7. Threshold sensitivity (validation, selected gamma): F1 and latency
# ------------------------------------------------------------
sg = sweep_cd[sweep_cd['gamma'] == CD_GAMMA]
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
for ax, col, cmap, title in [(axes[0], 'f1', 'viridis', 'F1 (validation)'),
                             (axes[1], 'latency_ms', 'magma_r', 'Mean latency (ms)')]:
    piv = sg.pivot(index='tau_legit', columns='tau_phish', values=col).sort_index(ascending=False)
    sns.heatmap(piv, cmap=cmap, ax=ax, cbar_kws={'label': title},
                xticklabels=2, yticklabels=2)
    # mark the selected thresholds
    xi = list(piv.columns).index(CD_TP); yi = list(piv.index).index(CD_TL)
    ax.add_patch(plt.Rectangle((xi, yi), 1, 1, fill=False, ec='red', lw=2.5))
    ax.set_xlabel('tau_phish'); ax.set_ylabel('tau_legit'); ax.set_title(title)
fig.suptitle(f'Cascade threshold sensitivity (gamma = {CD_GAMMA}), red box = selected')
save(fig, 'G7_threshold_sensitivity')

# ------------------------------------------------------------
# G8. Error examples (qualitative analysis): FN and FP
# ------------------------------------------------------------
fn_idx = np.where((y_test == 1) & (pred_prop == 0))[0][:4]
fp_idx = np.where((y_test == 0) & (pred_prop == 1))[0][:4]
if len(fn_idx) + len(fp_idx) > 0:
    fig, axes = plt.subplots(2, 4, figsize=(18, 7))
    for r, (idxs, title) in enumerate([(fn_idx, 'False negative (missed phishing)'),
                                       (fp_idx, 'False positive (flagged benign)')]):
        for c in range(4):
            ax = axes[r, c]; ax.axis('off')
            if c < len(idxs):
                i = idxs[c]
                ax.imshow(X224_test[i])
                url_short = df_test.loc[i, 'url'][:45] + ('...' if len(df_test.loc[i, 'url']) > 45 else '')
                ax.set_title(f'{url_short}\np={p_prop[i]:.2f} | exit at stage {st_prop[i]}', fontsize=8)
        axes[r, 0].text(-0.08, 0.5, title, transform=axes[r, 0].transAxes, rotation=90,
                        va='center', ha='right', fontsize=11)
    fig.suptitle('Error examples of the proposed method (test)')
    save(fig, 'G8_error_examples')

print('\nAll figures are saved in', FIG_DIR)